# Competitors

- radius is enough or do we need more?
- more columns?

In [ ]:
# %pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import requests
import pandas as pd
import time

import os
from dotenv import load_dotenv

load_dotenv("../.env")          # path from notebooks/ up to project root
API_KEY = os.getenv("GOOGLE_API_KEY")

print("Key loaded:", API_KEY is not None)   # never print the key itself


AREAS = {
    "Baneshwor": (27.69396, 85.33738),
    "New Road":  (27.70200, 85.30743),
    "Koteshwor": (27.68333, 85.35000),
    "Bhaktapur durbar square": (27.67203, 85.42811),
    "Patan durbar square":     (27.67340, 85.32500),
    "Boudha stupa": (27.72139, 85.36194),
    "Pulchowk": (27.6787,85.3175),
    "Durbar Marg": (27.71261, 85.31797),
    "Kirtipur": (27.67806, 85.27694),
}

TYPES = ["restaurant", "cafe", "bar", "bakery", "meal_takeaway", "coffee_shop"]

RADIUS = 1500
URL = "https://places.googleapis.com/v1/places:searchText"
MAX_PAGES = 5  # Text Search (New) pages 20 results at a time, up to 60 total

FIELDS = ",".join([
    "places.id", "places.displayName", "places.formattedAddress",
    "places.location", "places.rating", "places.userRatingCount",
    "places.priceLevel", "places.types", "places.primaryType", "places.reviews",
    "nextPageToken",
])

def search(lat, lng, place_type):
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": FIELDS,
    }
    all_places = []
    page_token = None
    for _ in range(MAX_PAGES):
        body = {
            "textQuery": place_type,
            "includedType": place_type,
            "locationBias": {
                "circle": {
                    "center": {"latitude": lat, "longitude": lng},
                    "radius": RADIUS,
                }
            },
            "maxResultCount": 20,
        }
        if page_token:
            body["pageToken"] = page_token

        r = requests.post(URL, headers=headers, json=body)
        if r.status_code != 200:
            print("  ERROR", r.status_code, r.text[:200])
            break

        data = r.json()
        all_places.extend(data.get("places", []))

        page_token = data.get("nextPageToken")
        if not page_token:
            break
        time.sleep(2)  # a freshly issued page token needs a moment before it's valid

    return all_places

rows = []
for area, (lat, lng) in AREAS.items():
    for t in TYPES:
        print(f"Searching {area} - {t} ...")
        results = search(lat, lng, t)
        print(f"  got {len(results)}")
        for p in results:
            reviews = p.get("reviews", [])
            review_texts = " ||| ".join(
                rv.get("text", {}).get("text", "") for rv in reviews
            )
            rows.append({
                "place_id": p.get("id"),
                "restaurant_name": p.get("displayName", {}).get("text"),
                "address": p.get("formattedAddress"),
                "latitude": p.get("location", {}).get("latitude"),
                "longitude": p.get("location", {}).get("longitude"),
                "restaurant_rating": p.get("rating"),
                "user_rating_count": p.get("userRatingCount"),
                "price_level": p.get("priceLevel"),
                "primary_type": p.get("primaryType"),
                "all_types": "|".join(p.get("types", [])),
                "search_area": area,
                "searched_as": t,
                "reviews": review_texts,
            })
        time.sleep(1)

df = pd.DataFrame(rows)
print("\nTotal rows before dedupe:", len(df))
df = df.drop_duplicates(subset="place_id").reset_index(drop=True)
print("Total unique places:", len(df))
print()
print(df["search_area"].value_counts())

df.to_csv("../data/raw_data/restaurants_raw_multiarea.csv", index=False)
print("\nSaved -> restaurants_raw_multiarea.csv")
df

Key loaded: True
Searching Baneshwor - restaurant ...
  got 60
Searching Baneshwor - cafe ...
  got 40
Searching Baneshwor - bar ...
  got 18
Searching Baneshwor - bakery ...
  got 60
Searching Baneshwor - meal_takeaway ...
  got 0
Searching Baneshwor - coffee_shop ...
  got 60
Searching New Road - restaurant ...
  got 60
Searching New Road - cafe ...
  got 46
Searching New Road - bar ...
  got 60
Searching New Road - bakery ...
  got 60
Searching New Road - meal_takeaway ...
  got 0
Searching New Road - coffee_shop ...
  got 60
Searching Koteshwor - restaurant ...
  got 60
Searching Koteshwor - cafe ...
  got 48
Searching Koteshwor - bar ...
  got 38
Searching Koteshwor - bakery ...
  got 60
Searching Koteshwor - meal_takeaway ...
  got 0
Searching Koteshwor - coffee_shop ...
  got 60
Searching Bhaktapur durbar square - restaurant ...
  got 60
Searching Bhaktapur durbar square - cafe ...
  got 46
Searching Bhaktapur durbar square - bar ...
  got 34
Searching Bhaktapur durbar square - 

,place_id,restaurant_name,address,latitude,longitude,restaurant_rating,user_rating_count,price_level,primary_type,all_types,search_area,searched_as,reviews
0,ChIJsZ5mY70Z6zkRRBczgL9a8ms,JAR - Just Another Restaurant,"Pipal Bot Marg, Kathmandu 56900, Nepal",27.699546,85.337687,4.3,443.0,PRICE_LEVEL_MODERATE,restaurant,restaurant|food|point_of_interest|establishment,Baneshwor,restaurant,This restaurant is located in Purano Baneshwor...
1,ChIJ5SO0EQ0Z6zkRTMnqUebrTqA,Drishya Lounge - Best Lounge in New Baneshwor,"Devkota Sadak, Kathmandu 44600, Nepal",27.692262,85.336472,4.4,1180.0,PRICE_LEVEL_EXPENSIVE,restaurant,restaurant|food|point_of_interest|establishment,Baneshwor,restaurant,I believe this is the best restaurant in Kathm...
2,ChIJTdMmLgAZ6zkR84zXOWwdgPE,Pink Putali Restaurant & Bar,"4 Chhakku Bakku Marg, Kathmandu 44600, Nepal",27.689064,85.334295,4.8,90.0,None,restaurant,restaurant|food|point_of_interest|establishment,Baneshwor,restaurant,"Newly opened R&B in New Baneshwor, Kathmandu!!..."
3,ChIJ3Ru72S8Z6zkR6JYG3tsYlGk,Munch N More Restaurant and Bar,"Janata Sadak, Kathmandu 44600, Nepal",27.681549,85.341340,4.9,60.0,None,restaurant,restaurant|food|point_of_interest|establishment,Baneshwor,restaurant,Food is satisfying at reasonable price and als...
4,ChIJJbL_SlQZ6zkRFWSjunPmuVU,27 Degree North Restaurant,"Shree Krishna Sadan, Chhakku Bakku Marg, Kathm...",27.688812,85.334080,4.7,43.0,None,restaurant,restaurant|food|point_of_interest|establishment,Baneshwor,restaurant,Live music every Friday with best service and ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1467,ChIJo0kCFbQZ6zkRFy3JaTRYXrE,Laya cafe,"M75M+X49, Kirtipur 44600, Nepal",27.659903,85.282868,4.7,7.0,None,coffee_shop,coffee_shop|cafe|point_of_interest|food_store|...,Kirtipur,coffee_shop,Laya cafe is my one of the best place...we pr...
1468,ChIJace5egAX6zkRwaMZGbdhvRE,Arabica Coffee,"M833+PXF, Karyabinayak 44600, Nepal",27.654317,85.304862,4.6,27.0,None,coffee_shop,coffee_shop|cafe|point_of_interest|food_store|...,Kirtipur,coffee_shop,This is a nice café with both outdoor and indo...
1469,ChIJX6NGPKQZ6zkRWoYQg3oeYeo,KAJU CAFE,"Panga Rd, Kirtipur 44600, Nepal",27.671268,85.279311,NaN,NaN,None,coffee_shop,coffee_shop|cafe|point_of_interest|food_store|...,Kirtipur,coffee_shop,
1470,ChIJE2U_5z8Z6zkR4_FR1IrUUS8,Universal Coffee Beans,"Dakchhinkali Road, 44600, Nepal",27.680025,85.297612,4.3,13.0,None,coffee_shop,coffee_shop|cafe|point_of_interest|food_store|...,Kirtipur,coffee_shop,🥂 ||| ||| ||| |||


# POIs

In [2]:
# pip install osmnx 
# run this in terminal

- schools haru primary, pre should we define
- also the number of students per school
- radius?

### FOR BANESHOWR

In [ ]:
import os
import math
import time
import requests
import pandas as pd


# ============================================================
# 1. CONFIGURATION
# ============================================================

CENTER = (27.69396, 85.33738)

# Keep only results within this final radius
MAIN_RADIUS = 1500

# Radius of every smaller search
# Reduce to 300 if many searches still return exactly 20
SMALL_RADIUS = 400

import os
from dotenv import load_dotenv

load_dotenv("../.env")          # path from notebooks/ up to project root
API_KEY = os.getenv("GOOGLE_API_KEY")


if not API_KEY:
    raise ValueError(
        "Google Maps API key not found. "
        "Set GOOGLE_MAPS_API_KEY before running this cell."
    )

URL = "https://places.googleapis.com/v1/places:searchNearby"


# ============================================================
# 2. POI TYPES
# ============================================================

POI_TYPES = {
    "school": ["school"],
    "college": ["university"],
    "hospital": ["hospital"],
    "clinic": ["medical_clinic"],
    "bank": ["bank"],
    "bus_stop": ["bus_stop"],
    "office": ["corporate_office"],
    "mall": ["shopping_mall"],
    "cinema": ["movie_theater"],
}


# ============================================================
# 3. DISTANCE CALCULATION
# ============================================================

def haversine_distance(lat1, lon1, lat2, lon2):
    """Return distance between two coordinates in meters."""

    earth_radius = 6_371_000

    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    value = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    return earth_radius * 2 * math.atan2(
        math.sqrt(value),
        math.sqrt(1 - value)
    )


# ============================================================
# 4. CONVERT METER OFFSETS TO COORDINATES
# ============================================================

def offset_coordinate(center, north_m, east_m):
    """Create a coordinate offset from the main center."""

    center_lat, center_lon = center

    new_lat = center_lat + north_m / 111_320

    new_lon = center_lon + (
        east_m
        / (
            111_320
            * math.cos(math.radians(center_lat))
        )
    )

    return new_lat, new_lon


# ============================================================
# 5. GENERATE SMALLER SEARCH CENTERS
# ============================================================

def generate_search_centers():
    """
    Generate overlapping search centers covering approximately
    the complete 1,500-meter area.
    """

    spacing = SMALL_RADIUS * 1.3

    offsets = []

    current_north = -MAIN_RADIUS

    while current_north <= MAIN_RADIUS:

        current_east = -MAIN_RADIUS

        while current_east <= MAIN_RADIUS:

            offset_distance = math.sqrt(
                current_north**2 + current_east**2
            )

            # Include centers whose smaller circles overlap
            # the original 1,500-meter circle
            if offset_distance <= MAIN_RADIUS + SMALL_RADIUS:
                offsets.append(
                    (current_north, current_east)
                )

            current_east += spacing

        current_north += spacing

    return [
        offset_coordinate(CENTER, north, east)
        for north, east in offsets
    ]


SEARCH_CENTERS = generate_search_centers()

print("Number of smaller search circles:", len(SEARCH_CENTERS))


# ============================================================
# 6. FETCH ONE POI TYPE FROM ONE SMALL CIRCLE
# ============================================================

def fetch_google_pois(google_types, search_center):
    """Fetch up to 20 results from one smaller search circle."""

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": (
            "places.id,"
            "places.displayName,"
            "places.location,"
            "places.primaryType,"
            "places.formattedAddress"
        )
    }

    payload = {
        "includedTypes": google_types,
        "maxResultCount": 20,
        "rankPreference": "DISTANCE",
        "locationRestriction": {
            "circle": {
                "center": {
                    "latitude": search_center[0],
                    "longitude": search_center[1]
                },
                "radius": float(SMALL_RADIUS)
            }
        }
    }

    response = requests.post(
        URL,
        headers=headers,
        json=payload,
        timeout=30
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Google Places API error "
            f"{response.status_code}: {response.text}"
        )

    return response.json().get("places", [])


# ============================================================
# 7. FETCH ALL CATEGORIES
# ============================================================

rows = []

for poi_name, google_types in POI_TYPES.items():

    print(f"\nFetching {poi_name}...")

    category_rows = []
    searches_reaching_limit = 0

    for search_number, search_center in enumerate(
        SEARCH_CENTERS,
        start=1
    ):

        try:
            places = fetch_google_pois(
                google_types=google_types,
                search_center=search_center
            )

            if len(places) == 20:
                searches_reaching_limit += 1

            for place in places:

                location = place.get("location", {})
                display_name = place.get("displayName", {})

                latitude = location.get("latitude")
                longitude = location.get("longitude")

                if latitude is None or longitude is None:
                    continue

                distance = haversine_distance(
                    CENTER[0],
                    CENTER[1],
                    latitude,
                    longitude
                )

                # Remove results outside the original
                # 1,500-meter Baneshwor circle
                if distance > MAIN_RADIUS:
                    continue

                category_rows.append({
                    "place_id": place.get("id"),
                    "poi_type": poi_name,
                    "name": display_name.get("text"),
                    "primary_type": place.get("primaryType"),
                    "formatted_address": place.get(
                        "formattedAddress"
                    ),
                    "latitude": latitude,
                    "longitude": longitude,
                    "distance_from_center_m": round(distance, 2)
                })

            print(
                f"\r  Completed search "
                f"{search_number}/{len(SEARCH_CENTERS)}",
                end=""
            )

            # Reduce the possibility of rate-limit errors
            time.sleep(0.1)

        except Exception as error:
            print(
                f"\n  Search {search_number} failed: {error}"
            )

    print()

    category_df = pd.DataFrame(category_rows)

    if not category_df.empty:
        category_df = (
            category_df
            .drop_duplicates(subset=["place_id"])
            .sort_values("distance_from_center_m")
            .reset_index(drop=True)
        )

        rows.extend(category_df.to_dict("records"))

    print(
        f"  Unique {poi_name}: {len(category_df)}"
    )

    if searches_reaching_limit > 0:
        print(
            f"  Warning: {searches_reaching_limit} searches "
            "returned the maximum 20 results."
        )


# ============================================================
# 8. CREATE FINAL DATAFRAME
# ============================================================

pois = pd.DataFrame(rows)

if not pois.empty:
    pois = (
        pois
        .drop_duplicates(
            subset=["place_id", "poi_type"]
        )
        .sort_values(
            ["poi_type", "distance_from_center_m"]
        )
        .reset_index(drop=True)
    )


# ============================================================
# 9. DISPLAY SUMMARY
# ============================================================

print("\n" + "=" * 50)
print("FINAL POI SUMMARY")
print("=" * 50)

print("\nTotal POI records:", len(pois))

if not pois.empty:
    print("\nPOI counts:")
    print(
        pois["poi_type"]
        .value_counts()
        .sort_index()
    )
else:
    print("No POIs found.")


# ============================================================
# 10. SAVE CSV
# ============================================================

output_path = "../data/raw_data/pois_baneshwor_google.csv"

os.makedirs(
    os.path.dirname(output_path),
    exist_ok=True
)

pois.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nSaved -> {output_path}")

pois

### FOR Koteshwor

In [15]:
import os
import math
import time
import requests
import pandas as pd


# ============================================================
# 1. CONFIGURATION
# ============================================================

CENTER = (27.68333, 85.35000)

# Keep only results within this final radius
MAIN_RADIUS = 1500

# Radius of every smaller search
# Reduce to 300 if many searches still return exactly 20
SMALL_RADIUS = 400

import os
from dotenv import load_dotenv

load_dotenv("../.env")          # path from notebooks/ up to project root
API_KEY = os.getenv("GOOGLE_API_KEY")


if not API_KEY:
    raise ValueError(
        "Google Maps API key not found. "
        "Set GOOGLE_MAPS_API_KEY before running this cell."
    )

URL = "https://places.googleapis.com/v1/places:searchNearby"


# ============================================================
# 2. POI TYPES
# ============================================================

POI_TYPES = {
    "school": ["school"],
    "college": ["university"],
    "hospital": ["hospital"],
    "clinic": ["medical_clinic"],
    "bank": ["bank"],
    "bus_stop": ["bus_stop"],
    "office": ["corporate_office"],
    "mall": ["shopping_mall"],
    "cinema": ["movie_theater"],
}


# ============================================================
# 3. DISTANCE CALCULATION
# ============================================================

def haversine_distance(lat1, lon1, lat2, lon2):
    """Return distance between two coordinates in meters."""

    earth_radius = 6_371_000

    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    value = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    return earth_radius * 2 * math.atan2(
        math.sqrt(value),
        math.sqrt(1 - value)
    )


# ============================================================
# 4. CONVERT METER OFFSETS TO COORDINATES
# ============================================================

def offset_coordinate(center, north_m, east_m):
    """Create a coordinate offset from the main center."""

    center_lat, center_lon = center

    new_lat = center_lat + north_m / 111_320

    new_lon = center_lon + (
        east_m
        / (
            111_320
            * math.cos(math.radians(center_lat))
        )
    )

    return new_lat, new_lon


# ============================================================
# 5. GENERATE SMALLER SEARCH CENTERS
# ============================================================

def generate_search_centers():
    """
    Generate overlapping search centers covering approximately
    the complete 1,500-meter area.
    """

    spacing = SMALL_RADIUS * 1.3

    offsets = []

    current_north = -MAIN_RADIUS

    while current_north <= MAIN_RADIUS:

        current_east = -MAIN_RADIUS

        while current_east <= MAIN_RADIUS:

            offset_distance = math.sqrt(
                current_north**2 + current_east**2
            )

            # Include centers whose smaller circles overlap
            # the original 1,500-meter circle
            if offset_distance <= MAIN_RADIUS + SMALL_RADIUS:
                offsets.append(
                    (current_north, current_east)
                )

            current_east += spacing

        current_north += spacing

    return [
        offset_coordinate(CENTER, north, east)
        for north, east in offsets
    ]


SEARCH_CENTERS = generate_search_centers()

print("Number of smaller search circles:", len(SEARCH_CENTERS))


# ============================================================
# 6. FETCH ONE POI TYPE FROM ONE SMALL CIRCLE
# ============================================================

def fetch_google_pois(google_types, search_center):
    """Fetch up to 20 results from one smaller search circle."""

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": (
            "places.id,"
            "places.displayName,"
            "places.location,"
            "places.primaryType,"
            "places.formattedAddress"
        )
    }

    payload = {
        "includedTypes": google_types,
        "maxResultCount": 20,
        "rankPreference": "DISTANCE",
        "locationRestriction": {
            "circle": {
                "center": {
                    "latitude": search_center[0],
                    "longitude": search_center[1]
                },
                "radius": float(SMALL_RADIUS)
            }
        }
    }

    response = requests.post(
        URL,
        headers=headers,
        json=payload,
        timeout=30
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Google Places API error "
            f"{response.status_code}: {response.text}"
        )

    return response.json().get("places", [])


# ============================================================
# 7. FETCH ALL CATEGORIES
# ============================================================

rows = []

for poi_name, google_types in POI_TYPES.items():

    print(f"\nFetching {poi_name}...")

    category_rows = []
    searches_reaching_limit = 0

    for search_number, search_center in enumerate(
        SEARCH_CENTERS,
        start=1
    ):

        try:
            places = fetch_google_pois(
                google_types=google_types,
                search_center=search_center
            )

            if len(places) == 20:
                searches_reaching_limit += 1

            for place in places:

                location = place.get("location", {})
                display_name = place.get("displayName", {})

                latitude = location.get("latitude")
                longitude = location.get("longitude")

                if latitude is None or longitude is None:
                    continue

                distance = haversine_distance(
                    CENTER[0],
                    CENTER[1],
                    latitude,
                    longitude
                )

                # Remove results outside the original
                # 1,500-meter Baneshwor circle
                if distance > MAIN_RADIUS:
                    continue

                category_rows.append({
                    "place_id": place.get("id"),
                    "poi_type": poi_name,
                    "name": display_name.get("text"),
                    "primary_type": place.get("primaryType"),
                    "formatted_address": place.get(
                        "formattedAddress"
                    ),
                    "latitude": latitude,
                    "longitude": longitude,
                    "distance_from_center_m": round(distance, 2)
                })

            print(
                f"\r  Completed search "
                f"{search_number}/{len(SEARCH_CENTERS)}",
                end=""
            )

            # Reduce the possibility of rate-limit errors
            time.sleep(0.1)

        except Exception as error:
            print(
                f"\n  Search {search_number} failed: {error}"
            )

    print()

    category_df = pd.DataFrame(category_rows)

    if not category_df.empty:
        category_df = (
            category_df
            .drop_duplicates(subset=["place_id"])
            .sort_values("distance_from_center_m")
            .reset_index(drop=True)
        )

        rows.extend(category_df.to_dict("records"))

    print(
        f"  Unique {poi_name}: {len(category_df)}"
    )

    if searches_reaching_limit > 0:
        print(
            f"  Warning: {searches_reaching_limit} searches "
            "returned the maximum 20 results."
        )


# ============================================================
# 8. CREATE FINAL DATAFRAME
# ============================================================

pois = pd.DataFrame(rows)

if not pois.empty:
    pois = (
        pois
        .drop_duplicates(
            subset=["place_id", "poi_type"]
        )
        .sort_values(
            ["poi_type", "distance_from_center_m"]
        )
        .reset_index(drop=True)
    )


# ============================================================
# 9. DISPLAY SUMMARY
# ============================================================

print("\n" + "=" * 50)
print("FINAL POI SUMMARY")
print("=" * 50)

print("\nTotal POI records:", len(pois))

if not pois.empty:
    print("\nPOI counts:")
    print(
        pois["poi_type"]
        .value_counts()
        .sort_index()
    )
else:
    print("No POIs found.")


# ============================================================
# 10. SAVE CSV
# ============================================================

output_path = "../data/raw_data/pois_koteshow_google.csv"

os.makedirs(
    os.path.dirname(output_path),
    exist_ok=True
)

pois.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nSaved -> {output_path}")

pois

Number of smaller search circles: 35

Fetching school...
  Completed search 6/35
  Search 7 failed: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
  Completed search 35/35
  Unique school: 151

Fetching college...
  Completed search 35/35
  Unique college: 100

Fetching hospital...
  Completed search 35/35
  Unique hospital: 88

Fetching clinic...
  Completed search 35/35
  Unique clinic: 198

Fetching bank...
  Completed search 35/35
  Unique bank: 96

Fetching bus_stop...
  Completed search 35/35
  Unique bus_stop: 17

Fetching office...
  Completed search 19/35
  Search 20 failed: HTTPSConnectionPool(host='places.googleapis.com', port=443): Read timed out. (read timeout=30)
  Completed search 21/35
  Search 22 failed: HTTPSConnectionPool(host='places.googleapis.com', port=443): Read timed out. (read timeout=30)

  Search 23 failed: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
  Completed se

,place_id,poi_type,name,primary_type,formatted_address,latitude,longitude,distance_from_center_m
0,ChIJzQ7UVsIZ6zkR8qwIgsqzQ1Q,bank,NIC ASIA Bank Koteshwor Branch,bank,"Ward no. 32, Kathmandu, 44600, Nepal",27.682737,85.349161,105.68
1,ChIJIfKwsYwZ6zkROVOYWn84Y9g,bank,Prabhu Bank Limited,bank,"M8MX+8HJ, Kathmandu 44600, Nepal",27.683334,85.348894,108.89
2,ChIJt1yKjvMZ6zkR4y26CrqnpIg,bank,Jyoti Bikash Bank Limited,bank,"32 Araniko Highway, Kathmandu 44600, Nepal",27.683323,85.348885,109.76
3,ChIJZc2UdgAZ6zkRtkGgL1OtY34,bank,Nepal Investment Mega Bank Limited,bank,"Kathmandu 44600, Nepal",27.681980,85.349242,167.62
4,ChIJVyggRfMZ6zkRF4ikgp4mldQ,bank,"Kumari Bank Limited, Koteshwor Branch",bank,"M8JX+RJ9, Kathmandu 44600, Nepal",27.682047,85.349102,167.80
...,...,...,...,...,...,...,...,...
1011,ChIJZVcjHeMb6zkRPvZuZpQ1q2g,school,Career Building International Academy,school,"M9R7+75F, CBIA street, Kathmandu 44600, Nepal",27.690487,85.362828,1492.92
1012,ChIJoTHBxUEb6zkRJeYFiuV8w_s,school,Gateway Montessori,school,"Unnamed Road, Kathmandu 44600, काठमाडौँ 44600,...",27.688484,85.364013,1494.01
1013,ChIJGR9B65UZ6zkREKVVzstMKL4,school,CAPITAL INT'L ACADEMY,school,"Bagbazar-31,Kathmandu, Nepal. (Just opposite t...",27.686975,85.335380,1495.51
1014,ChIJ4Wue9e8Z6zkRpq9jj-mJzKM,school,Gaurishankar English Boarding School (GS College),school,"M8CV+F9G, 44600, Nepal",27.671193,85.343440,1496.19


### For patan durbar square

In [16]:
import os
import math
import time
import requests
import pandas as pd


# ============================================================
# 1. CONFIGURATION
# ============================================================

CENTER = (27.67340, 85.32500)

# Keep only results within this final radius
MAIN_RADIUS = 1500

# Radius of every smaller search
# Reduce to 300 if many searches still return exactly 20
SMALL_RADIUS = 400

import os
from dotenv import load_dotenv

load_dotenv("../.env")          # path from notebooks/ up to project root
API_KEY = os.getenv("GOOGLE_API_KEY")


if not API_KEY:
    raise ValueError(
        "Google Maps API key not found. "
        "Set GOOGLE_MAPS_API_KEY before running this cell."
    )

URL = "https://places.googleapis.com/v1/places:searchNearby"


# ============================================================
# 2. POI TYPES
# ============================================================

POI_TYPES = {
    "school": ["school"],
    "college": ["university"],
    "hospital": ["hospital"],
    "clinic": ["medical_clinic"],
    "bank": ["bank"],
    "bus_stop": ["bus_stop"],
    "office": ["corporate_office"],
    "mall": ["shopping_mall"],
    "cinema": ["movie_theater"],
}


# ============================================================
# 3. DISTANCE CALCULATION
# ============================================================

def haversine_distance(lat1, lon1, lat2, lon2):
    """Return distance between two coordinates in meters."""

    earth_radius = 6_371_000

    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    value = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    return earth_radius * 2 * math.atan2(
        math.sqrt(value),
        math.sqrt(1 - value)
    )


# ============================================================
# 4. CONVERT METER OFFSETS TO COORDINATES
# ============================================================

def offset_coordinate(center, north_m, east_m):
    """Create a coordinate offset from the main center."""

    center_lat, center_lon = center

    new_lat = center_lat + north_m / 111_320

    new_lon = center_lon + (
        east_m
        / (
            111_320
            * math.cos(math.radians(center_lat))
        )
    )

    return new_lat, new_lon


# ============================================================
# 5. GENERATE SMALLER SEARCH CENTERS
# ============================================================

def generate_search_centers():
    """
    Generate overlapping search centers covering approximately
    the complete 1,500-meter area.
    """

    spacing = SMALL_RADIUS * 1.3

    offsets = []

    current_north = -MAIN_RADIUS

    while current_north <= MAIN_RADIUS:

        current_east = -MAIN_RADIUS

        while current_east <= MAIN_RADIUS:

            offset_distance = math.sqrt(
                current_north**2 + current_east**2
            )

            # Include centers whose smaller circles overlap
            # the original 1,500-meter circle
            if offset_distance <= MAIN_RADIUS + SMALL_RADIUS:
                offsets.append(
                    (current_north, current_east)
                )

            current_east += spacing

        current_north += spacing

    return [
        offset_coordinate(CENTER, north, east)
        for north, east in offsets
    ]


SEARCH_CENTERS = generate_search_centers()

print("Number of smaller search circles:", len(SEARCH_CENTERS))


# ============================================================
# 6. FETCH ONE POI TYPE FROM ONE SMALL CIRCLE
# ============================================================

def fetch_google_pois(google_types, search_center):
    """Fetch up to 20 results from one smaller search circle."""

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": (
            "places.id,"
            "places.displayName,"
            "places.location,"
            "places.primaryType,"
            "places.formattedAddress"
        )
    }

    payload = {
        "includedTypes": google_types,
        "maxResultCount": 20,
        "rankPreference": "DISTANCE",
        "locationRestriction": {
            "circle": {
                "center": {
                    "latitude": search_center[0],
                    "longitude": search_center[1]
                },
                "radius": float(SMALL_RADIUS)
            }
        }
    }

    response = requests.post(
        URL,
        headers=headers,
        json=payload,
        timeout=30
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Google Places API error "
            f"{response.status_code}: {response.text}"
        )

    return response.json().get("places", [])


# ============================================================
# 7. FETCH ALL CATEGORIES
# ============================================================

rows = []

for poi_name, google_types in POI_TYPES.items():

    print(f"\nFetching {poi_name}...")

    category_rows = []
    searches_reaching_limit = 0

    for search_number, search_center in enumerate(
        SEARCH_CENTERS,
        start=1
    ):

        try:
            places = fetch_google_pois(
                google_types=google_types,
                search_center=search_center
            )

            if len(places) == 20:
                searches_reaching_limit += 1

            for place in places:

                location = place.get("location", {})
                display_name = place.get("displayName", {})

                latitude = location.get("latitude")
                longitude = location.get("longitude")

                if latitude is None or longitude is None:
                    continue

                distance = haversine_distance(
                    CENTER[0],
                    CENTER[1],
                    latitude,
                    longitude
                )

                # Remove results outside the original
                # 1,500-meter Baneshwor circle
                if distance > MAIN_RADIUS:
                    continue

                category_rows.append({
                    "place_id": place.get("id"),
                    "poi_type": poi_name,
                    "name": display_name.get("text"),
                    "primary_type": place.get("primaryType"),
                    "formatted_address": place.get(
                        "formattedAddress"
                    ),
                    "latitude": latitude,
                    "longitude": longitude,
                    "distance_from_center_m": round(distance, 2)
                })

            print(
                f"\r  Completed search "
                f"{search_number}/{len(SEARCH_CENTERS)}",
                end=""
            )

            # Reduce the possibility of rate-limit errors
            time.sleep(0.1)

        except Exception as error:
            print(
                f"\n  Search {search_number} failed: {error}"
            )

    print()

    category_df = pd.DataFrame(category_rows)

    if not category_df.empty:
        category_df = (
            category_df
            .drop_duplicates(subset=["place_id"])
            .sort_values("distance_from_center_m")
            .reset_index(drop=True)
        )

        rows.extend(category_df.to_dict("records"))

    print(
        f"  Unique {poi_name}: {len(category_df)}"
    )

    if searches_reaching_limit > 0:
        print(
            f"  Warning: {searches_reaching_limit} searches "
            "returned the maximum 20 results."
        )


# ============================================================
# 8. CREATE FINAL DATAFRAME
# ============================================================

pois = pd.DataFrame(rows)

if not pois.empty:
    pois = (
        pois
        .drop_duplicates(
            subset=["place_id", "poi_type"]
        )
        .sort_values(
            ["poi_type", "distance_from_center_m"]
        )
        .reset_index(drop=True)
    )


# ============================================================
# 9. DISPLAY SUMMARY
# ============================================================

print("\n" + "=" * 50)
print("FINAL POI SUMMARY")
print("=" * 50)

print("\nTotal POI records:", len(pois))

if not pois.empty:
    print("\nPOI counts:")
    print(
        pois["poi_type"]
        .value_counts()
        .sort_index()
    )
else:
    print("No POIs found.")


# ============================================================
# 10. SAVE CSV
# ============================================================

output_path = "../data/raw_data/pois_patan_google.csv"

os.makedirs(
    os.path.dirname(output_path),
    exist_ok=True
)

pois.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nSaved -> {output_path}")

pois

Number of smaller search circles: 35

Fetching school...
  Completed search 35/35
  Unique school: 189

Fetching college...
  Completed search 35/35
  Unique college: 136

Fetching hospital...
  Completed search 35/35
  Unique hospital: 101

Fetching clinic...
  Completed search 35/35
  Unique clinic: 248

Fetching bank...
  Completed search 35/35
  Unique bank: 186

Fetching bus_stop...
  Completed search 35/35
  Unique bus_stop: 27

Fetching office...
  Completed search 35/35
  Unique office: 421

Fetching mall...
  Completed search 35/35
  Unique mall: 112

Fetching cinema...
  Completed search 35/35
  Unique cinema: 6

FINAL POI SUMMARY

Total POI records: 1426

POI counts:
poi_type
bank        186
bus_stop     27
cinema        6
clinic      248
college     136
hospital    101
mall        112
office      421
school      189
Name: count, dtype: int64

Saved -> ../data/raw_data/pois_patan_google.csv


,place_id,poi_type,name,primary_type,formatted_address,latitude,longitude,distance_from_center_m
0,ChIJs5XdmsUZ6zkR6ihsak2jvvU,bank,Layeku SACCOS,bank,"M8FF+HVM, मङ्गल बजार रोड, Lalitpur 44700, Nepal",27.673957,85.324683,69.36
1,ChIJnXAjgMUZ6zkRS1qJsbz6pXg,bank,Sikharadip SACCOS,bank,"M8FF+9JF, महापाल रोड, Lalitpur 44600, Nepal",27.673430,85.324054,93.22
2,ChIJp_S8PwAZ6zkRM3pmWSJz_zA,bank,Global IME Bank Limited,bank,"Lalitpur Patan-16, ललितपुर 44700, Nepal",27.673600,85.323786,121.63
3,ChIJdc-0GAgZ6zkRP-EcLc2ilf0,bank,Nible bank,bank,"M8FF+VVG, Kwalakhu Nhoolan Marg, Lalitpur 4470...",27.674632,85.324687,140.47
4,ChIJl7DSYooZ6zkR8oaZrd1QLZQ,bank,Manikeshav Narayan SACCOS,bank,"M8FG+V7V, Bangalamukhi-Shankhamul Rd, Lalitpur...",27.674694,85.325750,161.68
...,...,...,...,...,...,...,...,...
1421,ChIJm615kdQZ6zkRCCg9QBigRmw,school,Machhapuhhre International Pre School,preschool,"M878+75R, Ring Rd, Lalitpur 44700, Nepal",27.663226,85.315388,1475.08
1422,ChIJj4mK4tEZ6zkRbGsCWAfyDlw,school,Kidzee Satdobato,school,"Deepawali Marg, Lalitpur 44700, Nepal",27.660170,85.326361,1477.17
1423,ChIJnbVOA7YZ6zkRXEdPQhOIDEI,school,Merge Home Tuition,educational_institution,"Shree Nagar Marg, Kathmandu 44600, Nepal",27.682990,85.335634,1494.53
1424,ChIJKf2J6iwY6zkRhC_fe1phrmE,school,Kidland School,school,"रिङ्ग रोड, Lalitpur 44600, Nepal",27.666038,85.312245,1499.29


### For pulchowk

In [17]:
import os
import math
import time
import requests
import pandas as pd


# ============================================================
# 1. CONFIGURATION
# ============================================================

CENTER = (27.6787,85.3175)

# Keep only results within this final radius
MAIN_RADIUS = 1500

# Radius of every smaller search
# Reduce to 300 if many searches still return exactly 20
SMALL_RADIUS = 400

import os
from dotenv import load_dotenv

load_dotenv("../.env")          # path from notebooks/ up to project root
API_KEY = os.getenv("GOOGLE_API_KEY")


if not API_KEY:
    raise ValueError(
        "Google Maps API key not found. "
        "Set GOOGLE_MAPS_API_KEY before running this cell."
    )

URL = "https://places.googleapis.com/v1/places:searchNearby"


# ============================================================
# 2. POI TYPES
# ============================================================

POI_TYPES = {
    "school": ["school"],
    "college": ["university"],
    "hospital": ["hospital"],
    "clinic": ["medical_clinic"],
    "bank": ["bank"],
    "bus_stop": ["bus_stop"],
    "office": ["corporate_office"],
    "mall": ["shopping_mall"],
    "cinema": ["movie_theater"],
}


# ============================================================
# 3. DISTANCE CALCULATION
# ============================================================

def haversine_distance(lat1, lon1, lat2, lon2):
    """Return distance between two coordinates in meters."""

    earth_radius = 6_371_000

    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    value = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    return earth_radius * 2 * math.atan2(
        math.sqrt(value),
        math.sqrt(1 - value)
    )


# ============================================================
# 4. CONVERT METER OFFSETS TO COORDINATES
# ============================================================

def offset_coordinate(center, north_m, east_m):
    """Create a coordinate offset from the main center."""

    center_lat, center_lon = center

    new_lat = center_lat + north_m / 111_320

    new_lon = center_lon + (
        east_m
        / (
            111_320
            * math.cos(math.radians(center_lat))
        )
    )

    return new_lat, new_lon


# ============================================================
# 5. GENERATE SMALLER SEARCH CENTERS
# ============================================================

def generate_search_centers():
    """
    Generate overlapping search centers covering approximately
    the complete 1,500-meter area.
    """

    spacing = SMALL_RADIUS * 1.3

    offsets = []

    current_north = -MAIN_RADIUS

    while current_north <= MAIN_RADIUS:

        current_east = -MAIN_RADIUS

        while current_east <= MAIN_RADIUS:

            offset_distance = math.sqrt(
                current_north**2 + current_east**2
            )

            # Include centers whose smaller circles overlap
            # the original 1,500-meter circle
            if offset_distance <= MAIN_RADIUS + SMALL_RADIUS:
                offsets.append(
                    (current_north, current_east)
                )

            current_east += spacing

        current_north += spacing

    return [
        offset_coordinate(CENTER, north, east)
        for north, east in offsets
    ]


SEARCH_CENTERS = generate_search_centers()

print("Number of smaller search circles:", len(SEARCH_CENTERS))


# ============================================================
# 6. FETCH ONE POI TYPE FROM ONE SMALL CIRCLE
# ============================================================

def fetch_google_pois(google_types, search_center):
    """Fetch up to 20 results from one smaller search circle."""

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": (
            "places.id,"
            "places.displayName,"
            "places.location,"
            "places.primaryType,"
            "places.formattedAddress"
        )
    }

    payload = {
        "includedTypes": google_types,
        "maxResultCount": 20,
        "rankPreference": "DISTANCE",
        "locationRestriction": {
            "circle": {
                "center": {
                    "latitude": search_center[0],
                    "longitude": search_center[1]
                },
                "radius": float(SMALL_RADIUS)
            }
        }
    }

    response = requests.post(
        URL,
        headers=headers,
        json=payload,
        timeout=30
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Google Places API error "
            f"{response.status_code}: {response.text}"
        )

    return response.json().get("places", [])


# ============================================================
# 7. FETCH ALL CATEGORIES
# ============================================================

rows = []

for poi_name, google_types in POI_TYPES.items():

    print(f"\nFetching {poi_name}...")

    category_rows = []
    searches_reaching_limit = 0

    for search_number, search_center in enumerate(
        SEARCH_CENTERS,
        start=1
    ):

        try:
            places = fetch_google_pois(
                google_types=google_types,
                search_center=search_center
            )

            if len(places) == 20:
                searches_reaching_limit += 1

            for place in places:

                location = place.get("location", {})
                display_name = place.get("displayName", {})

                latitude = location.get("latitude")
                longitude = location.get("longitude")

                if latitude is None or longitude is None:
                    continue

                distance = haversine_distance(
                    CENTER[0],
                    CENTER[1],
                    latitude,
                    longitude
                )

                # Remove results outside the original
                # 1,500-meter Baneshwor circle
                if distance > MAIN_RADIUS:
                    continue

                category_rows.append({
                    "place_id": place.get("id"),
                    "poi_type": poi_name,
                    "name": display_name.get("text"),
                    "primary_type": place.get("primaryType"),
                    "formatted_address": place.get(
                        "formattedAddress"
                    ),
                    "latitude": latitude,
                    "longitude": longitude,
                    "distance_from_center_m": round(distance, 2)
                })

            print(
                f"\r  Completed search "
                f"{search_number}/{len(SEARCH_CENTERS)}",
                end=""
            )

            # Reduce the possibility of rate-limit errors
            time.sleep(0.1)

        except Exception as error:
            print(
                f"\n  Search {search_number} failed: {error}"
            )

    print()

    category_df = pd.DataFrame(category_rows)

    if not category_df.empty:
        category_df = (
            category_df
            .drop_duplicates(subset=["place_id"])
            .sort_values("distance_from_center_m")
            .reset_index(drop=True)
        )

        rows.extend(category_df.to_dict("records"))

    print(
        f"  Unique {poi_name}: {len(category_df)}"
    )

    if searches_reaching_limit > 0:
        print(
            f"  Warning: {searches_reaching_limit} searches "
            "returned the maximum 20 results."
        )


# ============================================================
# 8. CREATE FINAL DATAFRAME
# ============================================================

pois = pd.DataFrame(rows)

if not pois.empty:
    pois = (
        pois
        .drop_duplicates(
            subset=["place_id", "poi_type"]
        )
        .sort_values(
            ["poi_type", "distance_from_center_m"]
        )
        .reset_index(drop=True)
    )


# ============================================================
# 9. DISPLAY SUMMARY
# ============================================================

print("\n" + "=" * 50)
print("FINAL POI SUMMARY")
print("=" * 50)

print("\nTotal POI records:", len(pois))

if not pois.empty:
    print("\nPOI counts:")
    print(
        pois["poi_type"]
        .value_counts()
        .sort_index()
    )
else:
    print("No POIs found.")


# ============================================================
# 10. SAVE CSV
# ============================================================

output_path = "../data/raw_data/pois_pulchowk_google.csv"

os.makedirs(
    os.path.dirname(output_path),
    exist_ok=True
)

pois.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nSaved -> {output_path}")

pois

Number of smaller search circles: 35

Fetching school...
  Completed search 35/35
  Unique school: 188

Fetching college...
  Completed search 35/35
  Unique college: 143

Fetching hospital...
  Completed search 35/35
  Unique hospital: 101

Fetching clinic...
  Completed search 35/35
  Unique clinic: 275

Fetching bank...
  Completed search 35/35
  Unique bank: 169

Fetching bus_stop...
  Completed search 35/35
  Unique bus_stop: 16

Fetching office...
  Completed search 35/35
  Unique office: 486

Fetching mall...
  Completed search 35/35
  Unique mall: 101

Fetching cinema...
  Completed search 35/35
  Unique cinema: 3

FINAL POI SUMMARY

Total POI records: 1482

POI counts:
poi_type
bank        169
bus_stop     16
cinema        3
clinic      275
college     143
hospital    101
mall        101
office      486
school      188
Name: count, dtype: int64

Saved -> ../data/raw_data/pois_pulchowk_google.csv


,place_id,poi_type,name,primary_type,formatted_address,latitude,longitude,distance_from_center_m
0,ChIJHcnTU8oZ6zkRbjFcomkEkoI,bank,NIC ASIA Bank Pulchowk Branch,bank,"M8H8+PMR, Lalitpur 44700, Nepal",27.679319,85.316668,107.00
1,ChIJ60ZI68oZ6zkRbWk3kOtF3rg,bank,Everest Bank Ltd. Pulchowk Branch,bank,"M8H8+6J6, Lalitpur 44700, Nepal",27.678036,85.316501,123.00
2,ChIJVdgYqMsZ6zkRB9oI-2B1cMs,bank,Pragati SACCOS,bank,"M8H8+8FG, Lalitpur 44700, Nepal",27.678301,85.316214,134.17
3,ChIJJT0mM7QZ6zkR9fN5xo3LTg4,bank,Machhapuchchhre Bank Pulchowk,bank,"M8G9+X2M, Lalitpur 44600, Nepal",27.677439,85.317587,140.48
4,ChIJrZTWRMoZ6zkR-2ccvm9lhGE,bank,Guheswori Merchant Banking & Finance Ltd .,bank,"Harihar Bhawan Marg, Lalitpur 44700, Nepal",27.679846,85.316839,143.14
...,...,...,...,...,...,...,...,...
1477,ChIJB-3wD8MZ6zkRFLCSOY7u0tc,school,Early Kids Home,school,"M8FJ+594, Lalitpur 44700, Nepal",27.672882,85.330964,1475.24
1478,ChIJw9KePdEZ6zkRk1W6xyjvYr0,school,Lalitpur Madhyamik Vidyalaya (LMV),secondary_school,"M88C+75G, Lalitpur 44700, Nepal",27.665678,85.320459,1477.03
1479,ChIJMzJhpaQZ6zkR9QqZE0nPG-s,school,Technical Training & Research Institute (TTRI),school,"M899+WX8, Lalitpur 44600, Nepal",27.665486,85.319328,1480.33
1480,ChIJyZzUzy0Y6zkRMTt6FTktCAU,school,Mahindra Bhrikuti High School,secondary_school,"M895+2MC, Lalitpur 44600, Nepal",27.667558,85.309130,1488.07


### For kirtipur

In [19]:
import os
import math
import time
import requests
import pandas as pd


# ============================================================
# 1. CONFIGURATION
# ============================================================

CENTER = (27.67806, 85.27694)

# Keep only results within this final radius
MAIN_RADIUS = 1500

# Radius of every smaller search
# Reduce to 300 if many searches still return exactly 20
SMALL_RADIUS = 400

import os
from dotenv import load_dotenv

load_dotenv("../.env")          # path from notebooks/ up to project root
API_KEY = os.getenv("GOOGLE_API_KEY")


if not API_KEY:
    raise ValueError(
        "Google Maps API key not found. "
        "Set GOOGLE_MAPS_API_KEY before running this cell."
    )

URL = "https://places.googleapis.com/v1/places:searchNearby"


# ============================================================
# 2. POI TYPES
# ============================================================

POI_TYPES = {
    "school": ["school"],
    "college": ["university"],
    "hospital": ["hospital"],
    "clinic": ["medical_clinic"],
    "bank": ["bank"],
    "bus_stop": ["bus_stop"],
    "office": ["corporate_office"],
    "mall": ["shopping_mall"],
    "cinema": ["movie_theater"],
}


# ============================================================
# 3. DISTANCE CALCULATION
# ============================================================

def haversine_distance(lat1, lon1, lat2, lon2):
    """Return distance between two coordinates in meters."""

    earth_radius = 6_371_000

    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    value = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    return earth_radius * 2 * math.atan2(
        math.sqrt(value),
        math.sqrt(1 - value)
    )


# ============================================================
# 4. CONVERT METER OFFSETS TO COORDINATES
# ============================================================

def offset_coordinate(center, north_m, east_m):
    """Create a coordinate offset from the main center."""

    center_lat, center_lon = center

    new_lat = center_lat + north_m / 111_320

    new_lon = center_lon + (
        east_m
        / (
            111_320
            * math.cos(math.radians(center_lat))
        )
    )

    return new_lat, new_lon


# ============================================================
# 5. GENERATE SMALLER SEARCH CENTERS
# ============================================================

def generate_search_centers():
    """
    Generate overlapping search centers covering approximately
    the complete 1,500-meter area.
    """

    spacing = SMALL_RADIUS * 1.3

    offsets = []

    current_north = -MAIN_RADIUS

    while current_north <= MAIN_RADIUS:

        current_east = -MAIN_RADIUS

        while current_east <= MAIN_RADIUS:

            offset_distance = math.sqrt(
                current_north**2 + current_east**2
            )

            # Include centers whose smaller circles overlap
            # the original 1,500-meter circle
            if offset_distance <= MAIN_RADIUS + SMALL_RADIUS:
                offsets.append(
                    (current_north, current_east)
                )

            current_east += spacing

        current_north += spacing

    return [
        offset_coordinate(CENTER, north, east)
        for north, east in offsets
    ]


SEARCH_CENTERS = generate_search_centers()

print("Number of smaller search circles:", len(SEARCH_CENTERS))


# ============================================================
# 6. FETCH ONE POI TYPE FROM ONE SMALL CIRCLE
# ============================================================

def fetch_google_pois(google_types, search_center):
    """Fetch up to 20 results from one smaller search circle."""

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": (
            "places.id,"
            "places.displayName,"
            "places.location,"
            "places.primaryType,"
            "places.formattedAddress"
        )
    }

    payload = {
        "includedTypes": google_types,
        "maxResultCount": 20,
        "rankPreference": "DISTANCE",
        "locationRestriction": {
            "circle": {
                "center": {
                    "latitude": search_center[0],
                    "longitude": search_center[1]
                },
                "radius": float(SMALL_RADIUS)
            }
        }
    }

    response = requests.post(
        URL,
        headers=headers,
        json=payload,
        timeout=30
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Google Places API error "
            f"{response.status_code}: {response.text}"
        )

    return response.json().get("places", [])


# ============================================================
# 7. FETCH ALL CATEGORIES
# ============================================================

rows = []

for poi_name, google_types in POI_TYPES.items():

    print(f"\nFetching {poi_name}...")

    category_rows = []
    searches_reaching_limit = 0

    for search_number, search_center in enumerate(
        SEARCH_CENTERS,
        start=1
    ):

        try:
            places = fetch_google_pois(
                google_types=google_types,
                search_center=search_center
            )

            if len(places) == 20:
                searches_reaching_limit += 1

            for place in places:

                location = place.get("location", {})
                display_name = place.get("displayName", {})

                latitude = location.get("latitude")
                longitude = location.get("longitude")

                if latitude is None or longitude is None:
                    continue

                distance = haversine_distance(
                    CENTER[0],
                    CENTER[1],
                    latitude,
                    longitude
                )

                # Remove results outside the original
                # 1,500-meter Baneshwor circle
                if distance > MAIN_RADIUS:
                    continue

                category_rows.append({
                    "place_id": place.get("id"),
                    "poi_type": poi_name,
                    "name": display_name.get("text"),
                    "primary_type": place.get("primaryType"),
                    "formatted_address": place.get(
                        "formattedAddress"
                    ),
                    "latitude": latitude,
                    "longitude": longitude,
                    "distance_from_center_m": round(distance, 2)
                })

            print(
                f"\r  Completed search "
                f"{search_number}/{len(SEARCH_CENTERS)}",
                end=""
            )

            # Reduce the possibility of rate-limit errors
            time.sleep(0.1)

        except Exception as error:
            print(
                f"\n  Search {search_number} failed: {error}"
            )

    print()

    category_df = pd.DataFrame(category_rows)

    if not category_df.empty:
        category_df = (
            category_df
            .drop_duplicates(subset=["place_id"])
            .sort_values("distance_from_center_m")
            .reset_index(drop=True)
        )

        rows.extend(category_df.to_dict("records"))

    print(
        f"  Unique {poi_name}: {len(category_df)}"
    )

    if searches_reaching_limit > 0:
        print(
            f"  Warning: {searches_reaching_limit} searches "
            "returned the maximum 20 results."
        )


# ============================================================
# 8. CREATE FINAL DATAFRAME
# ============================================================

pois = pd.DataFrame(rows)

if not pois.empty:
    pois = (
        pois
        .drop_duplicates(
            subset=["place_id", "poi_type"]
        )
        .sort_values(
            ["poi_type", "distance_from_center_m"]
        )
        .reset_index(drop=True)
    )


# ============================================================
# 9. DISPLAY SUMMARY
# ============================================================

print("\n" + "=" * 50)
print("FINAL POI SUMMARY")
print("=" * 50)

print("\nTotal POI records:", len(pois))

if not pois.empty:
    print("\nPOI counts:")
    print(
        pois["poi_type"]
        .value_counts()
        .sort_index()
    )
else:
    print("No POIs found.")


# ============================================================
# 10. SAVE CSV
# ============================================================

output_path = "../data/raw_data/pois_kirtipur_google.csv"

os.makedirs(
    os.path.dirname(output_path),
    exist_ok=True
)

pois.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nSaved -> {output_path}")

pois

Number of smaller search circles: 35

Fetching school...
  Completed search 35/35
  Unique school: 75

Fetching college...
  Completed search 35/35
  Unique college: 42

Fetching hospital...
  Completed search 35/35
  Unique hospital: 29

Fetching clinic...
  Completed search 35/35
  Unique clinic: 42

Fetching bank...
  Completed search 35/35
  Unique bank: 30

Fetching bus_stop...
  Completed search 35/35
  Unique bus_stop: 3

Fetching office...
  Completed search 35/35
  Unique office: 83

Fetching mall...
  Completed search 35/35
  Unique mall: 20

Fetching cinema...
  Completed search 35/35
  Unique cinema: 3

FINAL POI SUMMARY

Total POI records: 327

POI counts:
poi_type
bank        30
bus_stop     3
cinema       3
clinic      42
college     42
hospital    29
mall        20
office      83
school      75
Name: count, dtype: int64

Saved -> ../data/raw_data/pois_kirtipur_google.csv


,place_id,poi_type,name,primary_type,formatted_address,latitude,longitude,distance_from_center_m
0,ChIJsdBStwoY6zkRdtkDOWruafk,bank,Kipu SACCOS,bank,"M7GG+X25, किर्तिपुर मार्ग, Kirtipur 44618, Nepal",27.677383,85.275104,195.87
1,ChIJ--vbkQ0Z6zkRhIsXye-_tXM,bank,Darshan SACCOS,bank,"M7GG+7JR, Ring Road, Kirtipur 44600, Nepal",27.675736,85.276507,261.91
2,ChIJSeWtVz0Z6zkRXvqcKX51JZ8,bank,Jyoti Bikash Bank,bank,"NEA Collection Branch, Kirtipur Ring Rd, Kirti...",27.675131,85.276840,325.85
3,ChIJCScaEhUZ6zkRRbfzT2jJcd0,bank,Nabil Bank Limited,bank,"M7FG+WV4, Kirtipur 44618, Nepal",27.674768,85.277203,366.94
4,ChIJj2kBqA4Y6zkRM5B1SejlV2E,bank,Agricultural Development Bank Ltd. Kritipur Br...,bank,"Kritipur ward no 9, M7FH+W54 Dobato, कीर्तिपुर...",27.674819,85.277932,373.38
...,...,...,...,...,...,...,...,...
322,ChIJK2E1CZwi6zkRkvYHlKwbev0,school,Prime English Secondary School,secondary_school,"M7P8+RWQ, Kalankisthan Rd, Chandragiri 44600, ...",27.687079,85.267363,1376.61
323,ChIJobNeuksj6zkRhg-3yYX6BkA,school,Bishnudevi Seconday School Gate,school,"M7J7+QGC, Chandragiri 44600, Nepal",27.682796,85.263823,1394.81
324,ChIJPcZlaQAZ6zkRFa0uP8J1H5M,school,जनपथ मार्ग,school,"M7RJ+823, Nagarjun 44600, Nepal",27.690690,85.279996,1436.32
325,ChIJrRZ_9XAY6zkR5XefhOil1C0,school,Janapath Secondary School,school,"M7RJ+72V, Nagarjun 44600, Nepal",27.690741,85.280058,1443.09


### for bkt durbar square and other places
- avi will run this, but the data set will be loaded here

# Road/ Accesibility

In [9]:
import os
import math
import osmnx as ox
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

AREAS = {
    "Baneshwor": (27.69396, 85.33738),
    "New Road":  (27.70200, 85.30743),
    "Koteshwor": (27.68333, 85.35000),
    "Bhaktapur durbar square": (27.67203, 85.42811),
    "Patan durbar square":     (27.67340, 85.32500),
    "Boudha stupa": (27.72139, 85.36194),
    "Pulchowk": (27.6787, 85.3175),
    "Durbar Marg": (27.71261, 85.31797),
    "Kirtipur": (27.67806, 85.27694),
}

RADIUS = 1500
OUTPUT_DIRECTORY = "../data/raw_data"
os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)

ox.settings.use_cache = True   # re-runs won't re-download

In [10]:
def process_area(area_name, center, radius=RADIUS):
    lat, lng = center

    # download + convert
    G = ox.graph_from_point(
        center_point=center, dist=radius,
        network_type="drive", simplify=True, retain_all=True
    )
    G_projected = ox.project_graph(G)
    nodes_p, edges_p = ox.graph_to_gdfs(G_projected, nodes=True, edges=True)
    projected_crs = edges_p.crs

    # exact circular study area
    center_pt = gpd.GeoSeries([Point(lng, lat)], crs="EPSG:4326")
    study_area = center_pt.to_crs(projected_crs).iloc[0].buffer(radius)

    # clip roads to the circle
    edges_clipped = edges_p.copy()
    edges_clipped["geometry"] = edges_clipped.geometry.intersection(study_area)
    edges_clipped = edges_clipped[~edges_clipped.geometry.is_empty].copy()
    edges_clipped["clipped_length_m"] = edges_clipped.geometry.length

    # drop duplicate directional edges
    edges_clipped = edges_clipped.reset_index()
    edges_clipped["pair_key"] = [
        (min(u, v), max(u, v), k)
        for u, v, k in zip(edges_clipped["u"], edges_clipped["v"], edges_clipped["key"])
    ]
    unique_edges = edges_clipped.drop_duplicates(subset=["pair_key"]).copy()

    # intersections inside the circle
    nodes_inside = nodes_p[nodes_p.geometry.within(study_area)].copy()

    # stats
    total_len_m = unique_edges["clipped_length_m"].sum()
    area_sqkm = math.pi * (radius / 1000) ** 2

    summary = {
        "area": area_name,
        "center_latitude": lat,
        "center_longitude": lng,
        "radius_m": radius,
        "area_sqkm": round(area_sqkm, 4),
        "intersection_count": len(nodes_inside),
        "road_segment_count": len(unique_edges),
        "total_road_length_m": round(total_len_m, 2),
        "road_density_m_per_sqkm": round(total_len_m / area_sqkm, 2),
        "road_density_km_per_sqkm": round((total_len_m / 1000) / area_sqkm, 2),
    }

    # back to lat/lng for saving
    roads_out = unique_edges.set_geometry("geometry").set_crs(projected_crs).to_crs("EPSG:4326")
    roads_out["geometry_wkt"] = roads_out.geometry.to_wkt()
    roads_out["area"] = area_name

    keep = [c for c in ["area", "u", "v", "key", "osmid", "name", "highway",
                        "oneway", "maxspeed", "clipped_length_m", "geometry_wkt"]
            if c in roads_out.columns]
    roads_csv = roads_out[keep].copy()

    nodes_out = nodes_inside.to_crs("EPSG:4326").reset_index()
    intersections_csv = pd.DataFrame({
        "area": area_name,
        "osmid": nodes_out["osmid"],
        "latitude": nodes_out.geometry.y,
        "longitude": nodes_out.geometry.x,
    })

    return roads_csv, intersections_csv, summary

In [11]:
all_roads, all_intersections, all_summaries = [], [], []

for area_name, center in AREAS.items():
    print(f"Processing {area_name} ...")
    try:
        roads, intersections, summary = process_area(area_name, center)
        all_roads.append(roads)
        all_intersections.append(intersections)
        all_summaries.append(summary)
        print(f"  {summary['intersection_count']} intersections, "
              f"{summary['total_road_length_m']/1000:.2f} km road")
    except Exception as e:
        print(f"  FAILED: {e}")

roads_df = pd.concat(all_roads, ignore_index=True)
intersections_df = pd.concat(all_intersections, ignore_index=True)
summary_df = pd.DataFrame(all_summaries)

summary_df

Processing Baneshwor ...
  1440 intersections, 140.06 km road
Processing New Road ...
  903 intersections, 101.74 km road
Processing Koteshwor ...
  1588 intersections, 137.12 km road
Processing Bhaktapur durbar square ...
  827 intersections, 96.12 km road
Processing Patan durbar square ...
  1081 intersections, 111.65 km road
Processing Boudha stupa ...
  1307 intersections, 122.57 km road
Processing Pulchowk ...
  1049 intersections, 109.57 km road
Processing Durbar Marg ...
  1010 intersections, 100.98 km road
Processing Kirtipur ...
  754 intersections, 83.26 km road


,area,center_latitude,center_longitude,radius_m,area_sqkm,intersection_count,road_segment_count,total_road_length_m,road_density_m_per_sqkm,road_density_km_per_sqkm
0,Baneshwor,27.69396,85.33738,1500,7.0686,1440,1903,140062.11,19814.74,19.81
1,New Road,27.70200,85.30743,1500,7.0686,903,1211,101740.94,14393.40,14.39
2,Koteshwor,27.68333,85.35000,1500,7.0686,1588,2102,137122.46,19398.86,19.40
3,Bhaktapur durbar square,27.67203,85.42811,1500,7.0686,827,1161,96116.25,13597.67,13.60
4,Patan durbar square,27.67340,85.32500,1500,7.0686,1081,1447,111649.02,15795.11,15.80
5,Boudha stupa,27.72139,85.36194,1500,7.0686,1307,1729,122568.03,17339.83,17.34
6,Pulchowk,27.67870,85.31750,1500,7.0686,1049,1376,109570.37,15501.04,15.50
7,Durbar Marg,27.71261,85.31797,1500,7.0686,1010,1304,100981.01,14285.89,14.29
8,Kirtipur,27.67806,85.27694,1500,7.0686,754,983,83258.40,11778.65,11.78


In [12]:
roads_df.to_csv(f"{OUTPUT_DIRECTORY}/roads_all_areas.csv", index=False)
intersections_df.to_csv(f"{OUTPUT_DIRECTORY}/intersections_all_areas.csv", index=False)
summary_df.to_csv(f"{OUTPUT_DIRECTORY}/road_summary_all_areas.csv", index=False)

print("Saved 3 files.")

Saved 3 files.


In [ ]:
# import os
# import math
# import osmnx as ox
# import pandas as pd
# import geopandas as gpd
# from shapely.geometry import Point


# # ============================================================
# # 1. CONFIGURATION
# # ============================================================

# CENTER = (27.69396, 85.33738)   # Baneshwor: latitude, longitude
# RADIUS = 1500                    # meters

# OUTPUT_DIRECTORY = "../data/raw_data"

# os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)


# # ============================================================
# # 2. DOWNLOAD THE ROAD NETWORK
# # ============================================================

# print("Downloading Baneshwor road network...")

# G = ox.graph_from_point(
#     center_point=CENTER,
#     dist=RADIUS,
#     network_type="drive",
#     simplify=True,
#     retain_all=True
# )

# print("Road network downloaded.")


# # ============================================================
# # 3. CONVERT GRAPH INTO TABLES
# # ============================================================

# nodes, edges = ox.graph_to_gdfs(
#     G,
#     nodes=True,
#     edges=True
# )

# print("Raw intersections/nodes:", len(nodes))
# print("Raw directed road edges:", len(edges))


# # ============================================================
# # 4. PROJECT DATA INTO A METRIC CRS
# # ============================================================

# # OSM data initially uses latitude/longitude.
# # Projection is needed for accurate meter-based calculations.

# G_projected = ox.project_graph(G)

# nodes_projected, edges_projected = ox.graph_to_gdfs(
#     G_projected,
#     nodes=True,
#     edges=True
# )

# projected_crs = edges_projected.crs

# print("Projected CRS:", projected_crs)


# # ============================================================
# # 5. CREATE THE EXACT 1,500-METER STUDY AREA
# # ============================================================

# center_point = gpd.GeoSeries(
#     [Point(CENTER[1], CENTER[0])],
#     crs="EPSG:4326"
# )

# center_projected = center_point.to_crs(projected_crs).iloc[0]

# study_area = center_projected.buffer(RADIUS)


# # ============================================================
# # 6. CLIP ROADS TO THE STUDY AREA
# # ============================================================

# edges_clipped = edges_projected.copy()

# edges_clipped["geometry"] = edges_clipped.geometry.intersection(
#     study_area
# )

# # Remove roads that do not intersect the circle
# edges_clipped = edges_clipped[
#     ~edges_clipped.geometry.is_empty
# ].copy()

# # Calculate clipped road length
# edges_clipped["clipped_length_m"] = (
#     edges_clipped.geometry.length
# )


# # ============================================================
# # 7. REMOVE DUPLICATE DIRECTIONAL EDGES
# # ============================================================

# # OSMnx graphs often contain one edge for each driving direction.
# # Without removing duplicates, two-way roads may be counted twice.

# edges_clipped["geometry_key"] = (
#     edges_clipped.geometry
#     .apply(lambda geometry: geometry.wkb_hex)
# )

# unique_edges = edges_clipped.drop_duplicates(
#     subset=["geometry_key"]
# ).copy()


# # ============================================================
# # 8. KEEP INTERSECTIONS INSIDE THE CIRCLE
# # ============================================================

# nodes_inside = nodes_projected[
#     nodes_projected.geometry.within(study_area)
# ].copy()


# # ============================================================
# # 9. CALCULATE ROAD STATISTICS
# # ============================================================

# intersection_count = len(nodes_inside)
# road_segment_count = len(unique_edges)

# total_road_length_m = unique_edges[
#     "clipped_length_m"
# ].sum()

# area_sqkm = math.pi * (RADIUS / 1000) ** 2

# road_density_m_per_sqkm = (
#     total_road_length_m / area_sqkm
# )

# road_density_km_per_sqkm = (
#     total_road_length_m / 1000
# ) / area_sqkm


# print("\nBANESHWOR ROAD-NETWORK SUMMARY")
# print("--------------------------------")

# print(
#     f"Intersections/nodes: {intersection_count}"
# )

# print(
#     f"Unique road segments: {road_segment_count}"
# )

# print(
#     f"Total road length: "
#     f"{total_road_length_m:.2f} meters"
# )

# print(
#     f"Total road length: "
#     f"{total_road_length_m / 1000:.2f} km"
# )

# print(
#     f"Study-area size: {area_sqkm:.2f} sq. km"
# )

# print(
#     f"Road density: "
#     f"{road_density_m_per_sqkm:.2f} m/sq. km"
# )

# print(
#     f"Road density: "
#     f"{road_density_km_per_sqkm:.2f} km/sq. km"
# )


# # ============================================================
# # 10. CONVERT BACK TO LATITUDE/LONGITUDE
# # ============================================================

# roads_output = unique_edges.to_crs("EPSG:4326")
# intersections_output = nodes_inside.to_crs("EPSG:4326")


# # ============================================================
# # 11. PREPARE ROAD CSV
# # ============================================================

# roads_output = roads_output.reset_index()

# roads_output["geometry_wkt"] = (
#     roads_output.geometry.to_wkt()
# )

# roads_columns = [
#     column
#     for column in [
#         "u",
#         "v",
#         "key",
#         "osmid",
#         "name",
#         "highway",
#         "oneway",
#         "maxspeed",
#         "clipped_length_m",
#         "geometry_wkt"
#     ]
#     if column in roads_output.columns
# ]

# roads_csv = roads_output[roads_columns].copy()


# # ============================================================
# # 12. PREPARE INTERSECTION CSV
# # ============================================================

# intersections_output = intersections_output.reset_index()

# intersections_csv = pd.DataFrame({
#     "osmid": intersections_output["osmid"],
#     "latitude": intersections_output.geometry.y,
#     "longitude": intersections_output.geometry.x
# })


# # ============================================================
# # 13. SAVE FILES
# # ============================================================

# roads_path = os.path.join(
#     OUTPUT_DIRECTORY,
#     "roads_baneshwor.csv"
# )

# intersections_path = os.path.join(
#     OUTPUT_DIRECTORY,
#     "intersections_baneshwor.csv"
# )

# summary_path = os.path.join(
#     OUTPUT_DIRECTORY,
#     "road_summary_baneshwor.csv"
# )


# roads_csv.to_csv(
#     roads_path,
#     index=False
# )

# intersections_csv.to_csv(
#     intersections_path,
#     index=False
# )


# road_summary = pd.DataFrame([{
#     "area": "baneshwor",
#     "center_latitude": CENTER[0],
#     "center_longitude": CENTER[1],
#     "radius_m": RADIUS,
#     "area_sqkm": round(area_sqkm, 4),
#     "intersection_count": intersection_count,
#     "road_segment_count": road_segment_count,
#     "total_road_length_m": round(
#         total_road_length_m,
#         2
#     ),
#     "road_density_m_per_sqkm": round(
#         road_density_m_per_sqkm,
#         2
#     ),
#     "road_density_km_per_sqkm": round(
#         road_density_km_per_sqkm,
#         2
#     )
# }])

# road_summary.to_csv(
#     summary_path,
#     index=False
# )


# print("\nSaved files:")
# print(" - roads_baneshwor.csv")
# print(" - intersections_baneshwor.csv")
# print(" - road_summary_baneshwor.csv")


# road_summary

Road network downloaded.
Raw intersections/nodes: 1731
Raw directed road edges: 4312
Projected CRS: EPSG:32645

BANESHWOR ROAD-NETWORK SUMMARY
--------------------------------
Intersections/nodes: 1440
Unique road segments: 3707
Total road length: 267852.37 meters
Total road length: 267.85 km
Study-area size: 7.07 sq. km
Road density: 37893.36 m/sq. km
Road density: 37.89 km/sq. km

Saved files:
 - roads_baneshwor.csv
 - intersections_baneshwor.csv
 - road_summary_baneshwor.csv


,area,center_latitude,center_longitude,radius_m,area_sqkm,intersection_count,road_segment_count,total_road_length_m,road_density_m_per_sqkm,road_density_km_per_sqkm
0,baneshwor,27.69396,85.33738,1500,7.0686,1440,3707,267852.37,37893.36,37.89


- how do i interpret this

# Population density with house/ building count


### Avi's part
- baneshwor 
- koteshwor
- bkt durbar square
- boudha
- durbar marg

### For Kirtipur

In [23]:
import os
import math
import osmnx as ox
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point


# ============================================================
# 1. CONFIGURATION
# ============================================================

AREA_NAME = "kirtipur"

# Approximate new_road center: latitude, longitude
CENTER = (27.67806, 85.27694)

RADIUS = 1500  # meters

OUTPUT_DIRECTORY = "../data/raw_data/house_density"

os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)


# ============================================================
# 2. FETCH BUILDINGS FROM OPENSTREETMAP
# ============================================================

print(f"Fetching buildings for {AREA_NAME.title()} from OpenStreetMap...")

buildings = ox.features_from_point(
    CENTER,
    tags={"building": True},
    dist=RADIUS
)

print("Raw building features found:", len(buildings))


# ============================================================
# 3. KEEP ONLY POLYGON BUILDINGS
# ============================================================

buildings = buildings[
    buildings.geometry.geom_type.isin(
        ["Polygon", "MultiPolygon"]
    )
].copy()

print("Polygon buildings found:", len(buildings))


# ============================================================
# 4. REMOVE DUPLICATE GEOMETRIES
# ============================================================

buildings["geometry_key"] = buildings.geometry.apply(
    lambda geometry: geometry.wkb_hex
)

buildings = buildings.drop_duplicates(
    subset=["geometry_key"]
).copy()

print("Unique building footprints:", len(buildings))


# ============================================================
# 5. PROJECT TO A METRIC CRS
# ============================================================

buildings_projected = ox.projection.project_gdf(buildings)

projected_crs = buildings_projected.crs

print("Projected CRS:", projected_crs)


# ============================================================
# 6. CREATE THE EXACT 1,500-METER STUDY CIRCLE
# ============================================================

center_point = gpd.GeoSeries(
    [Point(CENTER[1], CENTER[0])],
    crs="EPSG:4326"
)

center_projected = center_point.to_crs(
    projected_crs
).iloc[0]

study_area = center_projected.buffer(RADIUS)


# ============================================================
# 7. CLIP BUILDINGS TO THE STUDY AREA
# ============================================================

buildings_projected["clipped_geometry"] = (
    buildings_projected.geometry.intersection(study_area)
)

buildings_projected = buildings_projected[
    ~buildings_projected["clipped_geometry"].is_empty
].copy()

buildings_projected["building_area_sqm"] = (
    buildings_projected["clipped_geometry"].area
)

buildings_projected = buildings_projected[
    buildings_projected["building_area_sqm"] > 0
].copy()


# ============================================================
# 8. CREATE CENTROID LOCATIONS
# ============================================================

buildings_projected["centroid_geometry"] = (
    buildings_projected["clipped_geometry"].centroid
)

centroids = gpd.GeoSeries(
    buildings_projected["centroid_geometry"],
    crs=projected_crs
).to_crs("EPSG:4326")

buildings_projected["latitude"] = centroids.y.values
buildings_projected["longitude"] = centroids.x.values


# ============================================================
# 9. PREPARE BUILDING TYPES AND NAMES
# ============================================================

if "building" not in buildings_projected.columns:
    buildings_projected["building"] = "yes"

if "name" not in buildings_projected.columns:
    buildings_projected["name"] = None

buildings_projected["building_type"] = (
    buildings_projected["building"]
    .fillna("yes")
    .astype(str)
)

buildings_projected["name"] = (
    buildings_projected["name"]
    .where(buildings_projected["name"].notna(), None)
)


# ============================================================
# 10. CREATE OUTPUT TABLE
# ============================================================

bdf = pd.DataFrame({
    "building_type":
        buildings_projected["building_type"].values,

    "name":
        buildings_projected["name"].values,

    "latitude":
        buildings_projected["latitude"].values,

    "longitude":
        buildings_projected["longitude"].values,

    "building_area_sqm":
        buildings_projected["building_area_sqm"]
        .round(2)
        .values
})


# ============================================================
# 11. CALCULATE BUILDING-DENSITY STATISTICS
# ============================================================

study_area_sqkm = math.pi * (RADIUS / 1000) ** 2
study_area_sqm = math.pi * RADIUS ** 2

total_buildings = len(bdf)

total_building_footprint_sqm = (
    bdf["building_area_sqm"].sum()
)

building_density_per_sqkm = (
    total_buildings / study_area_sqkm
)

building_coverage_percentage = (
    total_building_footprint_sqm / study_area_sqm
) * 100

named_building_count = bdf["name"].notna().sum()
unnamed_building_count = bdf["name"].isna().sum()


# ============================================================
# 12. DISPLAY RESULTS
# ============================================================

print(f"\n{AREA_NAME.upper()} BUILDING SUMMARY")
print("----------------------------------")

print(f"Total buildings: {total_buildings}")
print(f"Named buildings: {named_building_count}")
print(f"Unnamed buildings: {unnamed_building_count}")
print(f"Study area: {study_area_sqkm:.2f} sq. km")

print(
    f"Building density: "
    f"{building_density_per_sqkm:.2f} buildings/sq. km"
)

print(
    f"Total building footprint: "
    f"{total_building_footprint_sqm:.2f} sq. m"
)

print(
    f"Approximate building coverage: "
    f"{building_coverage_percentage:.2f}%"
)

print("\nBuilding type breakdown:")

print(
    bdf["building_type"]
    .value_counts()
    .head(15)
)


# ============================================================
# 13. SAVE DETAILED BUILDING DATA
# ============================================================

building_output_path = os.path.join(
    OUTPUT_DIRECTORY,
    f"buildings_{AREA_NAME}.csv"
)

bdf.to_csv(
    building_output_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 14. SAVE SUMMARY FEATURES
# ============================================================

building_summary = pd.DataFrame([{
    "area": AREA_NAME,
    "center_latitude": CENTER[0],
    "center_longitude": CENTER[1],
    "radius_m": RADIUS,
    "study_area_sqkm": round(study_area_sqkm, 4),
    "building_count": total_buildings,
    "building_density_per_sqkm": round(
        building_density_per_sqkm,
        2
    ),
    "total_building_footprint_sqm": round(
        total_building_footprint_sqm,
        2
    ),
    "building_coverage_percentage": round(
        building_coverage_percentage,
        2
    ),
    "named_building_count": named_building_count,
    "unnamed_building_count": unnamed_building_count
}])

summary_output_path = os.path.join(
    OUTPUT_DIRECTORY,
    f"building_summary_{AREA_NAME}.csv"
)

building_summary.to_csv(
    summary_output_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 15. DISPLAY SAVED FILE INFORMATION
# ============================================================

print("\nSaved:")
print(f" - buildings_{AREA_NAME}.csv")
print(f" - building_summary_{AREA_NAME}.csv")

display(building_summary)
display(bdf.head())

bdf

Fetching buildings for Kirtipur from OpenStreetMap...
Raw building features found: 14572
Polygon buildings found: 14543
Unique building footprints: 14543
Projected CRS: EPSG:32645

KIRTIPUR BUILDING SUMMARY
----------------------------------
Total buildings: 11495
Named buildings: 126
Unnamed buildings: 11369
Study area: 7.07 sq. km
Building density: 1626.21 buildings/sq. km
Total building footprint: 1031089.45 sq. m
Approximate building coverage: 14.59%

Building type breakdown:
building_type
yes            10860
residential      189
house            176
greenhouse       102
school            67
commercial        21
mixed_use         14
college           14
hospital          10
university         8
educational        4
office             4
public             4
health_post        3
industrial         3
Name: count, dtype: int64

Saved:
 - buildings_kirtipur.csv
 - building_summary_kirtipur.csv


,area,center_latitude,center_longitude,radius_m,study_area_sqkm,building_count,building_density_per_sqkm,total_building_footprint_sqm,building_coverage_percentage,named_building_count,unnamed_building_count
0,kirtipur,27.67806,85.27694,1500,7.0686,11495,1626.21,1031089.45,14.59,126,11369


,building_type,name,latitude,longitude,building_area_sqm
0,college,Central Department of Computer Science and Inf...,27.681906,85.286523,3421.00
1,university,Gandhi Bhawan,27.680738,85.285320,782.21
2,yes,None,27.669676,85.277831,9.54
3,public,TU Central Library,27.681739,85.285122,3098.07
4,yes,None,27.688985,85.277420,357.65


,building_type,name,latitude,longitude,building_area_sqm
0,college,Central Department of Computer Science and Inf...,27.681906,85.286523,3421.00
1,university,Gandhi Bhawan,27.680738,85.285320,782.21
2,yes,None,27.669676,85.277831,9.54
3,public,TU Central Library,27.681739,85.285122,3098.07
4,yes,None,27.688985,85.277420,357.65
...,...,...,...,...,...
11490,yes,None,27.687003,85.267911,89.46
11491,yes,None,27.687092,85.267636,121.32
11492,yes,None,27.687110,85.267886,93.51
11493,yes,None,27.691340,85.276187,362.96


### For Pulchowk

In [22]:
import os
import math
import osmnx as ox
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point


# ============================================================
# 1. CONFIGURATION
# ============================================================

AREA_NAME = "pulchowk"

# Approximate new_road center: latitude, longitude
CENTER = (27.6787,85.3175)

RADIUS = 1500  # meters

OUTPUT_DIRECTORY = "../data/raw_data/house_density"

os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)


# ============================================================
# 2. FETCH BUILDINGS FROM OPENSTREETMAP
# ============================================================

print(f"Fetching buildings for {AREA_NAME.title()} from OpenStreetMap...")

buildings = ox.features_from_point(
    CENTER,
    tags={"building": True},
    dist=RADIUS
)

print("Raw building features found:", len(buildings))


# ============================================================
# 3. KEEP ONLY POLYGON BUILDINGS
# ============================================================

buildings = buildings[
    buildings.geometry.geom_type.isin(
        ["Polygon", "MultiPolygon"]
    )
].copy()

print("Polygon buildings found:", len(buildings))


# ============================================================
# 4. REMOVE DUPLICATE GEOMETRIES
# ============================================================

buildings["geometry_key"] = buildings.geometry.apply(
    lambda geometry: geometry.wkb_hex
)

buildings = buildings.drop_duplicates(
    subset=["geometry_key"]
).copy()

print("Unique building footprints:", len(buildings))


# ============================================================
# 5. PROJECT TO A METRIC CRS
# ============================================================

buildings_projected = ox.projection.project_gdf(buildings)

projected_crs = buildings_projected.crs

print("Projected CRS:", projected_crs)


# ============================================================
# 6. CREATE THE EXACT 1,500-METER STUDY CIRCLE
# ============================================================

center_point = gpd.GeoSeries(
    [Point(CENTER[1], CENTER[0])],
    crs="EPSG:4326"
)

center_projected = center_point.to_crs(
    projected_crs
).iloc[0]

study_area = center_projected.buffer(RADIUS)


# ============================================================
# 7. CLIP BUILDINGS TO THE STUDY AREA
# ============================================================

buildings_projected["clipped_geometry"] = (
    buildings_projected.geometry.intersection(study_area)
)

buildings_projected = buildings_projected[
    ~buildings_projected["clipped_geometry"].is_empty
].copy()

buildings_projected["building_area_sqm"] = (
    buildings_projected["clipped_geometry"].area
)

buildings_projected = buildings_projected[
    buildings_projected["building_area_sqm"] > 0
].copy()


# ============================================================
# 8. CREATE CENTROID LOCATIONS
# ============================================================

buildings_projected["centroid_geometry"] = (
    buildings_projected["clipped_geometry"].centroid
)

centroids = gpd.GeoSeries(
    buildings_projected["centroid_geometry"],
    crs=projected_crs
).to_crs("EPSG:4326")

buildings_projected["latitude"] = centroids.y.values
buildings_projected["longitude"] = centroids.x.values


# ============================================================
# 9. PREPARE BUILDING TYPES AND NAMES
# ============================================================

if "building" not in buildings_projected.columns:
    buildings_projected["building"] = "yes"

if "name" not in buildings_projected.columns:
    buildings_projected["name"] = None

buildings_projected["building_type"] = (
    buildings_projected["building"]
    .fillna("yes")
    .astype(str)
)

buildings_projected["name"] = (
    buildings_projected["name"]
    .where(buildings_projected["name"].notna(), None)
)


# ============================================================
# 10. CREATE OUTPUT TABLE
# ============================================================

bdf = pd.DataFrame({
    "building_type":
        buildings_projected["building_type"].values,

    "name":
        buildings_projected["name"].values,

    "latitude":
        buildings_projected["latitude"].values,

    "longitude":
        buildings_projected["longitude"].values,

    "building_area_sqm":
        buildings_projected["building_area_sqm"]
        .round(2)
        .values
})


# ============================================================
# 11. CALCULATE BUILDING-DENSITY STATISTICS
# ============================================================

study_area_sqkm = math.pi * (RADIUS / 1000) ** 2
study_area_sqm = math.pi * RADIUS ** 2

total_buildings = len(bdf)

total_building_footprint_sqm = (
    bdf["building_area_sqm"].sum()
)

building_density_per_sqkm = (
    total_buildings / study_area_sqkm
)

building_coverage_percentage = (
    total_building_footprint_sqm / study_area_sqm
) * 100

named_building_count = bdf["name"].notna().sum()
unnamed_building_count = bdf["name"].isna().sum()


# ============================================================
# 12. DISPLAY RESULTS
# ============================================================

print(f"\n{AREA_NAME.upper()} BUILDING SUMMARY")
print("----------------------------------")

print(f"Total buildings: {total_buildings}")
print(f"Named buildings: {named_building_count}")
print(f"Unnamed buildings: {unnamed_building_count}")
print(f"Study area: {study_area_sqkm:.2f} sq. km")

print(
    f"Building density: "
    f"{building_density_per_sqkm:.2f} buildings/sq. km"
)

print(
    f"Total building footprint: "
    f"{total_building_footprint_sqm:.2f} sq. m"
)

print(
    f"Approximate building coverage: "
    f"{building_coverage_percentage:.2f}%"
)

print("\nBuilding type breakdown:")

print(
    bdf["building_type"]
    .value_counts()
    .head(15)
)


# ============================================================
# 13. SAVE DETAILED BUILDING DATA
# ============================================================

building_output_path = os.path.join(
    OUTPUT_DIRECTORY,
    f"buildings_{AREA_NAME}.csv"
)

bdf.to_csv(
    building_output_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 14. SAVE SUMMARY FEATURES
# ============================================================

building_summary = pd.DataFrame([{
    "area": AREA_NAME,
    "center_latitude": CENTER[0],
    "center_longitude": CENTER[1],
    "radius_m": RADIUS,
    "study_area_sqkm": round(study_area_sqkm, 4),
    "building_count": total_buildings,
    "building_density_per_sqkm": round(
        building_density_per_sqkm,
        2
    ),
    "total_building_footprint_sqm": round(
        total_building_footprint_sqm,
        2
    ),
    "building_coverage_percentage": round(
        building_coverage_percentage,
        2
    ),
    "named_building_count": named_building_count,
    "unnamed_building_count": unnamed_building_count
}])

summary_output_path = os.path.join(
    OUTPUT_DIRECTORY,
    f"building_summary_{AREA_NAME}.csv"
)

building_summary.to_csv(
    summary_output_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 15. DISPLAY SAVED FILE INFORMATION
# ============================================================

print("\nSaved:")
print(f" - buildings_{AREA_NAME}.csv")
print(f" - building_summary_{AREA_NAME}.csv")

display(building_summary)
display(bdf.head())

bdf

Fetching buildings for Pulchowk from OpenStreetMap...
Raw building features found: 23065
Polygon buildings found: 23030
Unique building footprints: 23030
Projected CRS: EPSG:32645

PULCHOWK BUILDING SUMMARY
----------------------------------
Total buildings: 17691
Named buildings: 446
Unnamed buildings: 17245
Study area: 7.07 sq. km
Building density: 2502.76 buildings/sq. km
Total building footprint: 1989905.82 sq. m
Approximate building coverage: 28.15%

Building type breakdown:
building_type
yes             15668
residential       430
apartments        335
house             305
school            274
commercial        131
retail            127
college            88
mixed_use          77
office             68
hospital           31
educational        24
government         20
temple             19
kindergarten       14
Name: count, dtype: int64

Saved:
 - buildings_pulchowk.csv
 - building_summary_pulchowk.csv


,area,center_latitude,center_longitude,radius_m,study_area_sqkm,building_count,building_density_per_sqkm,total_building_footprint_sqm,building_coverage_percentage,named_building_count,unnamed_building_count
0,pulchowk,27.6787,85.3175,1500,7.0686,17691,2502.76,1989905.82,28.15,446,17245


,building_type,name,latitude,longitude,building_area_sqm
0,yes,Mani Keshar Chowk,27.673434,85.325347,657.81
1,yes,Sasto Bazaar,27.674820,85.323403,1709.23
2,school,None,27.676308,85.318494,970.81
3,yes,Pulchowk Campus Boys Hostel Block C,27.681833,85.323930,2348.73
4,school,None,27.685369,85.310715,1114.00


,building_type,name,latitude,longitude,building_area_sqm
0,yes,Mani Keshar Chowk,27.673434,85.325347,657.81
1,yes,Sasto Bazaar,27.674820,85.323403,1709.23
2,school,None,27.676308,85.318494,970.81
3,yes,Pulchowk Campus Boys Hostel Block C,27.681833,85.323930,2348.73
4,school,None,27.685369,85.310715,1114.00
...,...,...,...,...,...
17686,commercial,None,27.691490,85.317336,621.94
17687,yes,None,27.684935,85.319661,314.56
17688,yes,None,27.676618,85.327163,37.56
17689,college,"Department Of Civil Engineering, Pulchowk",27.682914,85.319486,1214.84


### for new road 

In [20]:
import os
import math
import osmnx as ox
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point


# ============================================================
# 1. CONFIGURATION
# ============================================================

AREA_NAME = "new_road"

# Approximate new_road center: latitude, longitude
CENTER = (27.70200, 85.30743)

RADIUS = 1500  # meters

OUTPUT_DIRECTORY = "../data/raw_data/house_density"

os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)


# ============================================================
# 2. FETCH BUILDINGS FROM OPENSTREETMAP
# ============================================================

print(f"Fetching buildings for {AREA_NAME.title()} from OpenStreetMap...")

buildings = ox.features_from_point(
    CENTER,
    tags={"building": True},
    dist=RADIUS
)

print("Raw building features found:", len(buildings))


# ============================================================
# 3. KEEP ONLY POLYGON BUILDINGS
# ============================================================

buildings = buildings[
    buildings.geometry.geom_type.isin(
        ["Polygon", "MultiPolygon"]
    )
].copy()

print("Polygon buildings found:", len(buildings))


# ============================================================
# 4. REMOVE DUPLICATE GEOMETRIES
# ============================================================

buildings["geometry_key"] = buildings.geometry.apply(
    lambda geometry: geometry.wkb_hex
)

buildings = buildings.drop_duplicates(
    subset=["geometry_key"]
).copy()

print("Unique building footprints:", len(buildings))


# ============================================================
# 5. PROJECT TO A METRIC CRS
# ============================================================

buildings_projected = ox.projection.project_gdf(buildings)

projected_crs = buildings_projected.crs

print("Projected CRS:", projected_crs)


# ============================================================
# 6. CREATE THE EXACT 1,500-METER STUDY CIRCLE
# ============================================================

center_point = gpd.GeoSeries(
    [Point(CENTER[1], CENTER[0])],
    crs="EPSG:4326"
)

center_projected = center_point.to_crs(
    projected_crs
).iloc[0]

study_area = center_projected.buffer(RADIUS)


# ============================================================
# 7. CLIP BUILDINGS TO THE STUDY AREA
# ============================================================

buildings_projected["clipped_geometry"] = (
    buildings_projected.geometry.intersection(study_area)
)

buildings_projected = buildings_projected[
    ~buildings_projected["clipped_geometry"].is_empty
].copy()

buildings_projected["building_area_sqm"] = (
    buildings_projected["clipped_geometry"].area
)

buildings_projected = buildings_projected[
    buildings_projected["building_area_sqm"] > 0
].copy()


# ============================================================
# 8. CREATE CENTROID LOCATIONS
# ============================================================

buildings_projected["centroid_geometry"] = (
    buildings_projected["clipped_geometry"].centroid
)

centroids = gpd.GeoSeries(
    buildings_projected["centroid_geometry"],
    crs=projected_crs
).to_crs("EPSG:4326")

buildings_projected["latitude"] = centroids.y.values
buildings_projected["longitude"] = centroids.x.values


# ============================================================
# 9. PREPARE BUILDING TYPES AND NAMES
# ============================================================

if "building" not in buildings_projected.columns:
    buildings_projected["building"] = "yes"

if "name" not in buildings_projected.columns:
    buildings_projected["name"] = None

buildings_projected["building_type"] = (
    buildings_projected["building"]
    .fillna("yes")
    .astype(str)
)

buildings_projected["name"] = (
    buildings_projected["name"]
    .where(buildings_projected["name"].notna(), None)
)


# ============================================================
# 10. CREATE OUTPUT TABLE
# ============================================================

bdf = pd.DataFrame({
    "building_type":
        buildings_projected["building_type"].values,

    "name":
        buildings_projected["name"].values,

    "latitude":
        buildings_projected["latitude"].values,

    "longitude":
        buildings_projected["longitude"].values,

    "building_area_sqm":
        buildings_projected["building_area_sqm"]
        .round(2)
        .values
})


# ============================================================
# 11. CALCULATE BUILDING-DENSITY STATISTICS
# ============================================================

study_area_sqkm = math.pi * (RADIUS / 1000) ** 2
study_area_sqm = math.pi * RADIUS ** 2

total_buildings = len(bdf)

total_building_footprint_sqm = (
    bdf["building_area_sqm"].sum()
)

building_density_per_sqkm = (
    total_buildings / study_area_sqkm
)

building_coverage_percentage = (
    total_building_footprint_sqm / study_area_sqm
) * 100

named_building_count = bdf["name"].notna().sum()
unnamed_building_count = bdf["name"].isna().sum()


# ============================================================
# 12. DISPLAY RESULTS
# ============================================================

print(f"\n{AREA_NAME.upper()} BUILDING SUMMARY")
print("----------------------------------")

print(f"Total buildings: {total_buildings}")
print(f"Named buildings: {named_building_count}")
print(f"Unnamed buildings: {unnamed_building_count}")
print(f"Study area: {study_area_sqkm:.2f} sq. km")

print(
    f"Building density: "
    f"{building_density_per_sqkm:.2f} buildings/sq. km"
)

print(
    f"Total building footprint: "
    f"{total_building_footprint_sqm:.2f} sq. m"
)

print(
    f"Approximate building coverage: "
    f"{building_coverage_percentage:.2f}%"
)

print("\nBuilding type breakdown:")

print(
    bdf["building_type"]
    .value_counts()
    .head(15)
)


# ============================================================
# 13. SAVE DETAILED BUILDING DATA
# ============================================================

building_output_path = os.path.join(
    OUTPUT_DIRECTORY,
    f"buildings_{AREA_NAME}.csv"
)

bdf.to_csv(
    building_output_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 14. SAVE SUMMARY FEATURES
# ============================================================

building_summary = pd.DataFrame([{
    "area": AREA_NAME,
    "center_latitude": CENTER[0],
    "center_longitude": CENTER[1],
    "radius_m": RADIUS,
    "study_area_sqkm": round(study_area_sqkm, 4),
    "building_count": total_buildings,
    "building_density_per_sqkm": round(
        building_density_per_sqkm,
        2
    ),
    "total_building_footprint_sqm": round(
        total_building_footprint_sqm,
        2
    ),
    "building_coverage_percentage": round(
        building_coverage_percentage,
        2
    ),
    "named_building_count": named_building_count,
    "unnamed_building_count": unnamed_building_count
}])

summary_output_path = os.path.join(
    OUTPUT_DIRECTORY,
    f"building_summary_{AREA_NAME}.csv"
)

building_summary.to_csv(
    summary_output_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 15. DISPLAY SAVED FILE INFORMATION
# ============================================================

print("\nSaved:")
print(f" - buildings_{AREA_NAME}.csv")
print(f" - building_summary_{AREA_NAME}.csv")

display(building_summary)
display(bdf.head())

bdf

Fetching buildings for New_Road from OpenStreetMap...
Raw building features found: 33467
Polygon buildings found: 33446
Unique building footprints: 33444
Projected CRS: EPSG:32645

NEW_ROAD BUILDING SUMMARY
----------------------------------
Total buildings: 28087
Named buildings: 481
Unnamed buildings: 27606
Study area: 7.07 sq. km
Building density: 3973.50 buildings/sq. km
Total building footprint: 2348601.39 sq. m
Approximate building coverage: 33.23%

Building type breakdown:
building_type
yes            26256
residential      413
house            276
mixed_use        241
school           228
apartments       163
commercial       115
retail           106
college           51
hotel             43
government        42
hospital          25
temple            21
office            20
educational       19
Name: count, dtype: int64

Saved:
 - buildings_new_road.csv
 - building_summary_new_road.csv


,area,center_latitude,center_longitude,radius_m,study_area_sqkm,building_count,building_density_per_sqkm,total_building_footprint_sqm,building_coverage_percentage,named_building_count,unnamed_building_count
0,new_road,27.702,85.30743,1500,7.0686,28087,3973.5,2348601.39,33.23,481,27606


,building_type,name,latitude,longitude,building_area_sqm
0,yes,Kathmandu Metropolitian City office,27.698928,85.311890,1924.09
1,yes,None,27.692586,85.314546,1720.43
2,Advertising_Agency,Himali Advertising Agency (P) Ltd,27.706044,85.318369,681.06
3,yes,Bijeshwori Temple,27.713936,85.300758,300.82
4,yes,Kumari House,27.703744,85.306465,457.82


,building_type,name,latitude,longitude,building_area_sqm
0,yes,Kathmandu Metropolitian City office,27.698928,85.311890,1924.09
1,yes,None,27.692586,85.314546,1720.43
2,Advertising_Agency,Himali Advertising Agency (P) Ltd,27.706044,85.318369,681.06
3,yes,Bijeshwori Temple,27.713936,85.300758,300.82
4,yes,Kumari House,27.703744,85.306465,457.82
...,...,...,...,...,...
28082,yes,None,27.691361,85.308684,112.02
28083,yes,None,27.701861,85.312838,51.83
28084,yes,None,27.701801,85.312820,52.69
28085,yes,None,27.707797,85.313044,31.68


### for Patan durbar square

In [21]:
import os
import math
import osmnx as ox
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point


# ============================================================
# 1. CONFIGURATION
# ============================================================

AREA_NAME = "patan"

# Approximate patan center: latitude, longitude
CENTER = (27.67340, 85.32500)

RADIUS = 1500  # meters

OUTPUT_DIRECTORY = "../data/raw_data/house_density"

os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)


# ============================================================
# 2. FETCH BUILDINGS FROM OPENSTREETMAP
# ============================================================

print(f"Fetching buildings for {AREA_NAME.title()} from OpenStreetMap...")

buildings = ox.features_from_point(
    CENTER,
    tags={"building": True},
    dist=RADIUS
)

print("Raw building features found:", len(buildings))


# ============================================================
# 3. KEEP ONLY POLYGON BUILDINGS
# ============================================================

buildings = buildings[
    buildings.geometry.geom_type.isin(
        ["Polygon", "MultiPolygon"]
    )
].copy()

print("Polygon buildings found:", len(buildings))


# ============================================================
# 4. REMOVE DUPLICATE GEOMETRIES
# ============================================================

buildings["geometry_key"] = buildings.geometry.apply(
    lambda geometry: geometry.wkb_hex
)

buildings = buildings.drop_duplicates(
    subset=["geometry_key"]
).copy()

print("Unique building footprints:", len(buildings))


# ============================================================
# 5. PROJECT TO A METRIC CRS
# ============================================================

buildings_projected = ox.projection.project_gdf(buildings)

projected_crs = buildings_projected.crs

print("Projected CRS:", projected_crs)


# ============================================================
# 6. CREATE THE EXACT 1,500-METER STUDY CIRCLE
# ============================================================

center_point = gpd.GeoSeries(
    [Point(CENTER[1], CENTER[0])],
    crs="EPSG:4326"
)

center_projected = center_point.to_crs(
    projected_crs
).iloc[0]

study_area = center_projected.buffer(RADIUS)


# ============================================================
# 7. CLIP BUILDINGS TO THE STUDY AREA
# ============================================================

buildings_projected["clipped_geometry"] = (
    buildings_projected.geometry.intersection(study_area)
)

buildings_projected = buildings_projected[
    ~buildings_projected["clipped_geometry"].is_empty
].copy()

buildings_projected["building_area_sqm"] = (
    buildings_projected["clipped_geometry"].area
)

buildings_projected = buildings_projected[
    buildings_projected["building_area_sqm"] > 0
].copy()


# ============================================================
# 8. CREATE CENTROID LOCATIONS
# ============================================================

buildings_projected["centroid_geometry"] = (
    buildings_projected["clipped_geometry"].centroid
)

centroids = gpd.GeoSeries(
    buildings_projected["centroid_geometry"],
    crs=projected_crs
).to_crs("EPSG:4326")

buildings_projected["latitude"] = centroids.y.values
buildings_projected["longitude"] = centroids.x.values


# ============================================================
# 9. PREPARE BUILDING TYPES AND NAMES
# ============================================================

if "building" not in buildings_projected.columns:
    buildings_projected["building"] = "yes"

if "name" not in buildings_projected.columns:
    buildings_projected["name"] = None

buildings_projected["building_type"] = (
    buildings_projected["building"]
    .fillna("yes")
    .astype(str)
)

buildings_projected["name"] = (
    buildings_projected["name"]
    .where(buildings_projected["name"].notna(), None)
)


# ============================================================
# 10. CREATE OUTPUT TABLE
# ============================================================

bdf = pd.DataFrame({
    "building_type":
        buildings_projected["building_type"].values,

    "name":
        buildings_projected["name"].values,

    "latitude":
        buildings_projected["latitude"].values,

    "longitude":
        buildings_projected["longitude"].values,

    "building_area_sqm":
        buildings_projected["building_area_sqm"]
        .round(2)
        .values
})


# ============================================================
# 11. CALCULATE BUILDING-DENSITY STATISTICS
# ============================================================

study_area_sqkm = math.pi * (RADIUS / 1000) ** 2
study_area_sqm = math.pi * RADIUS ** 2

total_buildings = len(bdf)

total_building_footprint_sqm = (
    bdf["building_area_sqm"].sum()
)

building_density_per_sqkm = (
    total_buildings / study_area_sqkm
)

building_coverage_percentage = (
    total_building_footprint_sqm / study_area_sqm
) * 100

named_building_count = bdf["name"].notna().sum()
unnamed_building_count = bdf["name"].isna().sum()


# ============================================================
# 12. DISPLAY RESULTS
# ============================================================

print(f"\n{AREA_NAME.upper()} BUILDING SUMMARY")
print("----------------------------------")

print(f"Total buildings: {total_buildings}")
print(f"Named buildings: {named_building_count}")
print(f"Unnamed buildings: {unnamed_building_count}")
print(f"Study area: {study_area_sqkm:.2f} sq. km")

print(
    f"Building density: "
    f"{building_density_per_sqkm:.2f} buildings/sq. km"
)

print(
    f"Total building footprint: "
    f"{total_building_footprint_sqm:.2f} sq. m"
)

print(
    f"Approximate building coverage: "
    f"{building_coverage_percentage:.2f}%"
)

print("\nBuilding type breakdown:")

print(
    bdf["building_type"]
    .value_counts()
    .head(15)
)


# ============================================================
# 13. SAVE DETAILED BUILDING DATA
# ============================================================

building_output_path = os.path.join(
    OUTPUT_DIRECTORY,
    f"buildings_{AREA_NAME}.csv"
)

bdf.to_csv(
    building_output_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 14. SAVE SUMMARY FEATURES
# ============================================================

building_summary = pd.DataFrame([{
    "area": AREA_NAME,
    "center_latitude": CENTER[0],
    "center_longitude": CENTER[1],
    "radius_m": RADIUS,
    "study_area_sqkm": round(study_area_sqkm, 4),
    "building_count": total_buildings,
    "building_density_per_sqkm": round(
        building_density_per_sqkm,
        2
    ),
    "total_building_footprint_sqm": round(
        total_building_footprint_sqm,
        2
    ),
    "building_coverage_percentage": round(
        building_coverage_percentage,
        2
    ),
    "named_building_count": named_building_count,
    "unnamed_building_count": unnamed_building_count
}])

summary_output_path = os.path.join(
    OUTPUT_DIRECTORY,
    f"building_summary_{AREA_NAME}.csv"
)

building_summary.to_csv(
    summary_output_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 15. DISPLAY SAVED FILE INFORMATION
# ============================================================

print("\nSaved:")
print(f" - buildings_{AREA_NAME}.csv")
print(f" - building_summary_{AREA_NAME}.csv")

display(building_summary)
display(bdf.head())

bdf

Fetching buildings for Patan from OpenStreetMap...
Raw building features found: 21828
Polygon buildings found: 21798
Unique building footprints: 21798
Projected CRS: EPSG:32645

PATAN BUILDING SUMMARY
----------------------------------
Total buildings: 17671
Named buildings: 396
Unnamed buildings: 17275
Study area: 7.07 sq. km
Building density: 2499.94 buildings/sq. km
Total building footprint: 1984939.89 sq. m
Approximate building coverage: 28.08%

Building type breakdown:
building_type
yes             16088
residential       342
school            293
apartments        238
house             225
college            91
commercial         87
retail             72
mixed_use          44
office             42
hospital           21
temple             19
government         15
kindergarten       11
educational        11
Name: count, dtype: int64

Saved:
 - buildings_patan.csv
 - building_summary_patan.csv


,area,center_latitude,center_longitude,radius_m,study_area_sqkm,building_count,building_density_per_sqkm,total_building_footprint_sqm,building_coverage_percentage,named_building_count,unnamed_building_count
0,patan,27.6734,85.325,1500,7.0686,17671,2499.94,1984939.89,28.08,396,17275


,building_type,name,latitude,longitude,building_area_sqm
0,yes,Mani Keshar Chowk,27.673434,85.325347,657.81
1,yes,Sasto Bazaar,27.674820,85.323403,1709.23
2,school,None,27.676308,85.318494,970.81
3,yes,Pulchowk Campus Boys Hostel Block C,27.681833,85.323930,2348.73
4,yes,None,27.670096,85.314104,1169.96


,building_type,name,latitude,longitude,building_area_sqm
0,yes,Mani Keshar Chowk,27.673434,85.325347,657.81
1,yes,Sasto Bazaar,27.674820,85.323403,1709.23
2,school,None,27.676308,85.318494,970.81
3,yes,Pulchowk Campus Boys Hostel Block C,27.681833,85.323930,2348.73
4,yes,None,27.670096,85.314104,1169.96
...,...,...,...,...,...
17666,yes,None,27.676618,85.327163,37.56
17667,yes,None,27.680274,85.334014,80.27
17668,yes,None,27.668508,85.334154,24.51
17669,college,"Department Of Civil Engineering, Pulchowk",27.682914,85.319486,1214.84


In [ ]:
import osmnx as ox
import pandas as pd
import time

ox.settings.use_cache = True
ox.settings.timeout = 300

AREAS = {
    "Baneshwor": (27.69396, 85.33738),
    "New Road":  (27.70200, 85.30743),
    "Koteshwor": (27.68333, 85.35000),
    "Bhaktapur": (27.68333, 85.41667),
    "Patan":     (27.67660, 85.32500),
    "Thamel":    (27.71540, 85.31090),
}
RADIUS = 1500

all_b = []
for area, center in AREAS.items():
    print(f"\nFetching buildings for {area} ...")
    for attempt in range(3):
        try:
            b = ox.features_from_point(center, tags={"building": True}, dist=RADIUS)
            rows = []
            for _, row in b.iterrows():
                geom = row.geometry.centroid
                rows.append({
                    "building_type": row.get("building", "yes"),
                    "name": row.get("name", None),
                    "latitude": geom.y,
                    "longitude": geom.x,
                    "search_area": area,
                })
            all_b.append(pd.DataFrame(rows))
            print(f"  got {len(rows)}")
            break
        except Exception as ex:
            print(f"  attempt {attempt+1} failed: {str(ex)[:70]}")
            if attempt < 2:
                print("  waiting 60s ...")
                time.sleep(60)
    time.sleep(30)

buildings = pd.concat(all_b, ignore_index=True)
before = len(buildings)
buildings = buildings.drop_duplicates(subset=["latitude","longitude"]).reset_index(drop=True)
print(f"\nRemoved {before-len(buildings)} duplicates from overlapping areas")
print("Total buildings:", len(buildings))
print()
print("BY AREA:")
print(buildings["search_area"].value_counts())

buildings.to_csv("../data/raw_data/buildings_multiarea.csv", index=False)
print("\nSaved -> buildings_multiarea.csv")
buildings

Fetching buildings for Baneshwor ...
  29559
Fetching buildings for New Road ...
  failed: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<HTTPSConnection(host='overpass-api.de', port=443) at 0x2777fe94410>, 'Connection to overpass-api.de timed out. (connect timeout=180)'))
Fetching buildings for Koteshwor ...
  failed: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<HTTPSConnection(host='overpass-api.de', port=443) at 0x2777fe94b90>, 'Connection to overpass-api.de timed out. (connect timeout=180)'))
Fetching buildings for Bhaktapur ...
  failed: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<HTTPSConnection(host='overpass-api.de', port=443) at 0x2777fe95090>, 'Connection to overpass-api.de timed out. (connect timeout=180)'))
Fe

,building_type,name,latitude,longitude,search_area
0,yes,Subekchya Hostel,27.687587,85.333969,Baneshwor
1,office,Planet Earth Solutions,27.686682,85.349605,Baneshwor
2,yes,डेनेब इन्टरनेशनल स्कुल,27.691599,85.331881,Baneshwor
3,yes,Federation of Contractors' Association of Nepal,27.693106,85.329014,Baneshwor
4,yes,National Investigation Department,27.693753,85.326209,Baneshwor
...,...,...,...,...,...
29554,yes,NaN,27.694561,85.335712,Baneshwor
29555,yes,NaN,27.684916,85.346461,Baneshwor
29556,yes,NaN,27.684897,85.346527,Baneshwor
29557,yes,NaN,27.687999,85.348562,Baneshwor


# MAP
- TO SEE WHAT 1500 RADIUS OF BANESHWOR COVERS

In [ ]:
# %pip install folium


   ------------- -------------------------- 1/3 [branca]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   ---------------------------------------- 3/3 [folium]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
import pandas as pd
import folium

df = pd.read_csv("../data/raw_data/restaurants_raw_multiarea.csv")

AREAS = {
    "Baneshwor": (27.69396, 85.33738),
    "New Road":  (27.70200, 85.30743),
    "Koteshwor": (27.68333, 85.35000),
    "Bhaktapur durbar square": (27.67203, 85.42811),
    "Patan durbar square":     (27.67340, 85.32500),
    "Boudha stupa": (27.72139, 85.36194),
    "Pulchowk": (27.6787, 85.3175),
    "Durbar Marg": (27.71261, 85.31797),
    "Kirtipur": (27.67806, 85.27694),
}

COLORS = {
    "Baneshwor": "blue",
    "New Road": "green",
    "Koteshwor": "orange",
    "Bhaktapur durbar square": "purple",
    "Patan durbar square": "darkred",
    "Boudha stupa": "cadetblue",
    "Pulchowk": "darkgreen",
    "Durbar Marg": "pink",
    "Kirtipur": "black",
}

RADIUS = 1500  # meters, same radius used during collection

# fit the map to bounds that cover every search circle, with padding for the circles themselves
lats = [lat for lat, lng in AREAS.values()]
lngs = [lng for lat, lng in AREAS.values()]
pad = 0.02  # degrees
bounds = [[min(lats) - pad, min(lngs) - pad], [max(lats) + pad, max(lngs) + pad]]

m = folium.Map()
m.fit_bounds(bounds)

# draw each search circle
for area, center in AREAS.items():
    folium.Circle(
        center, radius=RADIUS, color=COLORS[area],
        fill=True, fill_opacity=0.08, popup=area
    ).add_to(m)

# plot each restaurant, coloured by its search area
for _, r in df.iterrows():
    folium.CircleMarker(
        location=(r["latitude"], r["longitude"]),
        radius=3,
        color=COLORS.get(r["search_area"], "gray"),
        fill=True, fill_opacity=0.8,
        popup=f"{r['restaurant_name']} ({r['search_area']})"
    ).add_to(m)

m.save("../outputs/multiarea_map.html")
print("Saved -> outputs/multiarea_map.html")
m

Saved -> outputs/multiarea_map.html
